# UD3.02 — NumPy: algebra lineal, memoria y rendimiento

**Modulo 5073 · Programacion de Inteligencia Artificial · Curso 2026/27**
UD3 — NumPy y Pandas · 14 horas

Criterios 1.d y 2.b · Material de partida de la practica P3.1


## Objetivos de Aprendizaje

Al finalizar este notebook, serás capaz de:

- **Aplicar descomposiciones matriciales** (SVD, QR, eigenvalues) para análisis de datos y reducción de dimensionalidad
- **Comprender la diferencia** entre views y copies, y optimizar el uso de memoria en arrays
- **Utilizar técnicas de vectorización** para eliminar bucles y acelerar código hasta 100x
- **Implementar broadcasting avanzado** para operaciones complejas entre arrays multidimensionales
- **Aplicar NumPy en casos reales de IA** como procesamiento de imágenes y preparación de datos para machine learning

## Introducción

### Del NumPy Básico al NumPy Avanzado

En el notebook anterior aprendimos los fundamentos de NumPy: creación de arrays, indexación, broadcasting básico y operaciones matemáticas. Ahora vamos a profundizar en técnicas avanzadas que son esenciales para trabajar con datasets grandes y algoritmos de machine learning.

### ¿Por qué NumPy Avanzado?

Si NumPy básico es como aprender a conducir un coche, NumPy avanzado es como aprender a ajustar el motor, optimizar el consumo de combustible y realizar maniobras de competición. Necesitas este conocimiento cuando trabajas con:

- **Datasets de millones de registros** donde cada milisegundo cuenta
- **Modelos de deep learning** donde las operaciones matriciales son el 90% del tiempo de cómputo
- **Procesamiento de imágenes** con arrays 3D y 4D de gran tamaño
- **Análisis de componentes principales (PCA)** y reducción de dimensionalidad

### Temas Clave

1. **Álgebra Lineal Avanzada:** Descomposiciones matriciales que potencian algoritmos de ML
2. **Views vs Copies:** Cómo NumPy gestiona la memoria y cómo evitar copias innecesarias
3. **Vectorización:** Eliminar bucles para código hasta 100x más rápido
4. **Broadcasting Avanzado:** Operaciones complejas sin bucles explícitos
5. **Aplicaciones en IA:** Casos prácticos de procesamiento de imágenes y preparación de datos

### Configuración Inicial

In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# La visualizacion se estudia a fondo en la UD4. Aqui los graficos son solo
# una herramienta para mirar los datos: nada de estilos ni de bibliotecas extra.
np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print("numpy", np.__version__, "\u00b7 pandas", pd.__version__)


## 1. Álgebra Lineal Avanzada

### ¿Por qué Álgebra Lineal en IA?

El álgebra lineal es el lenguaje matemático del machine learning. Prácticamente todos los algoritmos modernos de IA se pueden expresar como operaciones matriciales:

- **Redes neuronales:** Cada capa es una multiplicación matricial seguida de una función de activación
- **PCA:** Utiliza descomposición en valores propios para reducir dimensionalidad
- **Sistemas de recomendación:** SVD (Singular Value Decomposition) para factorización de matrices
- **Regresión lineal:** Resuelve el sistema normal mediante descomposición QR o SVD

### 1.1. Descomposición en Valores Singulares (SVD)

#### ¿Qué es SVD y por qué es tan importante?

Imagina que tienes una foto de 1000×1000 píxeles (1 millón de números). **SVD te permite guardar solo la "esencia" de la imagen** usando muchos menos números. Es como si alguien te pidiera describir una película: en lugar de contar cada fotograma, describes lo más importante.

**Analogía del mundo real - Una orquesta:**
- Una orquesta tiene 100 músicos tocando
- El **primer valor singular** es como el director: captura el ritmo y tempo general (lo más importante)
- El **segundo valor singular** es como la sección de cuerdas: añade la melodía principal
- El **tercer valor singular** es como los metales: añade armonías
- Los valores singulares pequeños son como instrumentos de fondo: aportan detalles pero no son esenciales
- Si grabas solo al director + cuerdas + metales (primeros 3 valores), capturas el 90% de la música

#### ¿Qué hace SVD matemáticamente?

La **SVD** descompone cualquier matriz A en tres matrices:

$$A = U \Sigma V^T$$

**Explicación visual de cada componente:**

1. **U (Vectores Singulares Izquierdos):**
   - Son las "direcciones" en el espacio de las filas
   - En imágenes: representan patrones verticales (columnas)
   - Ejemplo: en una foto de caras, cada columna de U podría representar "forma de ojos", "forma de nariz", etc.

2. **Σ (Valores Singulares - la diagonal):**
   - Son números que indican **cuánto importa cada dirección**
   - Van ordenados de mayor a menor: el primero es el MÁS importante
   - Ejemplo: σ₁ = 100 (muy importante), σ₂ = 50 (importante), σ₃ = 10 (algo importante), σ₄ = 0.1 (casi nada)
   - **CLAVE:** Si un valor singular es pequeño, podemos ignorar esa componente sin perder mucha información

3. **Vᵀ (Vectores Singulares Derechos):**
   - Son las "direcciones" en el espacio de las columnas
   - En imágenes: representan patrones horizontales (filas)
   - Ejemplo: en una foto, cada fila de Vᵀ podría representar texturas o patrones de iluminación

#### ¿Por qué son importantes los valores singulares?

Los valores singulares te dicen **cuánta información contiene cada componente**:

- **Valores grandes** → Información importante, NO podemos ignorarla
- **Valores pequeños** → Detalles, ruido, podemos descartarlos sin perder mucho

**Ejemplo numérico:**
```
Valores singulares: [150, 80, 30, 5, 0.1, 0.01]

Interpretación:
- 150: ESENCIAL (captura el 60% de la información)
- 80: MUY IMPORTANTE (captura el 30%)
- 30: IMPORTANTE (captura el 8%)
- 5: Detalles (captura el 1.5%)
- 0.1, 0.01: Ruido (captura el 0.5%)

Si usamos solo los 3 primeros, conservamos el 98% de la información original.
```

#### Aplicaciones prácticas de SVD:

1. **Compresión de imágenes:**
   - Imagen original: 1000×1000 = 1,000,000 números
   - Con SVD (50 componentes): 50×(1000+1000+1) ≈ 100,000 números
   - ¡10 veces menos espacio! Y la imagen se ve casi igual

2. **Sistemas de recomendación (Netflix, Amazon):**
   - Matriz usuarios×películas (millones de valores desconocidos)
   - SVD encuentra patrones ocultos: "usuarios que les gusta acción", "películas románticas", etc.
   - Puede predecir qué películas te gustarán aunque nunca las hayas visto

3. **Reducción de ruido:**
   - Los valores singulares pequeños suelen representar ruido
   - Al descartarlos, limpiamos los datos automáticamente

4. **Análisis de texto (búsqueda semántica):**
   - Encuentra relaciones entre documentos aunque usen palabras diferentes
   - "coche" y "automóvil" quedan cerca en el espacio SVD

**En resumen:** SVD descompone datos complejos en componentes ordenadas por importancia, permitiéndonos quedarnos solo con lo esencial.

In [ ]:
# ============================================================
# EJEMPLO PRÁCTICO: Compresión de imagen usando SVD
# ============================================================
#
# OBJETIVO: Ver cómo SVD nos permite comprimir una imagen
# manteniendo la calidad visual
#
# PASOS:
# 1. Crear una "imagen" simulada (matriz de números)
# 2. Aplicar SVD para descomponerla
# 3. Ver qué información contiene cada componente
# ============================================================

# PASO 1: Crear imagen simulada
# --------------------------------
# En la realidad sería una foto real, pero aquí simulamos una matriz 20x20
# Cada número representa un nivel de gris (0=negro, 255=blanco)

rng = np.random.default_rng(seed=42)
imagen_original = rng.uniform(0, 255, size=(20, 20))

print("=" * 60)
print("IMAGEN ORIGINAL")
print("=" * 60)
print(f"Dimensiones: {imagen_original.shape} (20 filas × 20 columnas)")
print(f"Total de números: {imagen_original.size} números")
print(f"Tamaño en memoria: {imagen_original.nbytes} bytes")
print()

# PASO 2: Aplicar SVD - La magia comienza aquí
# ------------------------------------------------
# SVD descompone la imagen en 3 matrices: U, S (valores singulares), Vt
#
# Fórmula: imagen_original = U @ diag(S) @ Vt
#
# full_matrices=False significa que solo guardamos lo necesario

print("=" * 60)
print("APLICANDO SVD...")
print("=" * 60)

U, S, Vt = np.linalg.svd(imagen_original, full_matrices=False)

print("\n¡SVD completado! Veamos qué obtuvimos:\n")

# PASO 3: Entender las matrices resultantes
# -------------------------------------------
print("Matriz U (vectores singulares izquierdos):")
print(f" - Shape: {U.shape}")
print(f" - Contiene: Patrones de las FILAS de la imagen")
print(f" - Cada columna de U es un 'patrón base'")
print()

print("Vector S (valores singulares - LO MÁS IMPORTANTE):")
print(f" - Shape: {S.shape}")
print(f" - Contiene: {len(S)} valores que indican importancia")
print(f" - Están ordenados de MAYOR a MENOR")
print()

print("Matriz Vt (vectores singulares derechos transpuestos):")
print(f" - Shape: {Vt.shape}")
print(f" - Contiene: Patrones de las COLUMNAS de la imagen")
print(f" - Cada fila de Vt es un 'patrón base'")
print()

# PASO 4: Analizar los valores singulares
# -----------------------------------------
print("=" * 60)
print("ANÁLISIS DE VALORES SINGULARES")
print("=" * 60)
print(f"\nPrimeros 10 valores singulares:")
print(S[:10])
print()

# Calcular cuánta información captura cada componente
# La información se mide como el cuadrado del valor singular
energia = S ** 2
energia_total = np.sum(energia)
porcentaje_acumulado = np.cumsum(energia) / energia_total * 100

print("Información capturada (acumulada):")
print(f" - 1 componente: {porcentaje_acumulado[0]:.2f}%")
print(f" - 2 componentes: {porcentaje_acumulado[1]:.2f}%")
print(f" - 5 componentes: {porcentaje_acumulado[4]:.2f}%")
print(f" - 10 componentes: {porcentaje_acumulado[9]:.2f}%")
print(f" - 20 componentes: {porcentaje_acumulado[19]:.2f}%")
print()
print("Conclusión: Con solo 10 componentes (50% de los datos)")
print(f" capturamos el {porcentaje_acumulado[9]:.1f}% de la información!")
print("=" * 60)

In [ ]:
# ============================================================
# RECONSTRUCCIÓN DE IMAGEN CON DIFERENTES COMPONENTES
# ============================================================
#
# OBJETIVO: Ver visualmente cómo afecta el número de componentes
# a la calidad de la imagen reconstruida
#
# CONCEPTO CLAVE:
# - Más componentes = mejor calidad pero más espacio
# - Menos componentes = peor calidad pero menos espacio
# - El truco es encontrar el equilibrio perfecto
# ============================================================

# Lista de componentes que probaremos
componentes = [2, 5, 10, 20]

print("=" * 60)
print("RECONSTRUYENDO IMAGEN CON DIFERENTES COMPONENTES")
print("=" * 60)
print()

# Preparar visualización
fig, axes = plt.subplots(1, len(componentes) + 1, figsize=(18, 3))

# PRIMERA IMAGEN: Original (100% de componentes)
# ------------------------------------------------
axes[0].imshow(imagen_original, cmap='gray')
axes[0].set_title('Original\n(20 componentes = 100%)', fontsize=10, fontweight='bold')
axes[0].axis('off')

# SIGUIENTES IMÁGENES: Reconstrucciones con k componentes
# ---------------------------------------------------------
for idx, k in enumerate(componentes, start=1):
    
    print(f"Reconstruyendo con {k} componentes...")
    
    # PASO 1: Seleccionar solo los primeros k componentes
    # -----------------------------------------------------
    # De U tomamos las primeras k columnas: U[:, :k]
    # De S tomamos los primeros k valores: S[:k]
    # De Vt tomamos las primeras k filas: Vt[:k, :]
    
    U_k = U[:, :k] # Shape: (20, k)
    S_k = S[:k] # Shape: (k,)
    Vt_k = Vt[:k, :] # Shape: (k, 20)
    
    # PASO 2: Reconstruir la imagen
    # ------------------------------
    # Fórmula: imagen_reconstruida = U_k @ diag(S_k) @ Vt_k
    #
    # Explicación:
    # - U_k @ diag(S_k) multiplica los patrones por su importancia
    # - Luego @ Vt_k combina todo para reconstruir la imagen
    
    imagen_reconstruida = U_k @ np.diag(S_k) @ Vt_k
    
    # PASO 3: Calcular métricas de compresión
    # ----------------------------------------
    # ¿Cuánto espacio ahorramos?
    # Original: 20×20 = 400 números
    # Comprimido: k×(20+1+20) = k×41 números
    
    numeros_originales = imagen_original.size
    numeros_comprimidos = k * (U.shape[0] + 1 + Vt.shape[1])
    ratio_compresion = numeros_comprimidos / numeros_originales * 100
    
    # ¿Cuánto error tenemos?
    # Error = diferencia entre original y reconstruida
    error = np.linalg.norm(imagen_original - imagen_reconstruida)
    error_relativo = error / np.linalg.norm(imagen_original) * 100
    
    print(f" - Tamaño: {ratio_compresion:.1f}% del original")
    print(f" - Error relativo: {error_relativo:.2f}%")
    print()
    
    # PASO 4: Visualizar
    # -------------------
    axes[idx].imshow(imagen_reconstruida, cmap='gray')
    axes[idx].set_title(
        f'{k} componentes\n({ratio_compresion:.0f}% tamaño | {error_relativo:.1f}% error)',
        fontsize=10
    )
    axes[idx].axis('off')

plt.suptitle('Compresión de Imagen con SVD - Comparación Visual',
             fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

print("=" * 60)
print("CONCLUSIONES:")
print("=" * 60)
print("1. Con SOLO 2 componentes (10% del espacio) ya vemos la estructura")
print("2. Con 5 componentes (25% del espacio) la imagen es reconocible")
print("3. Con 10 componentes (50% del espacio) apenas se nota diferencia")
print("4. Con 20 componentes (100%) es la imagen original exacta")
print()
print("APLICACIÓN PRÁCTICA:")
print("- Si necesitas enviar esta imagen por WhatsApp en zona sin WiFi")
print(" → Usa 5 componentes: 4x más rápido de enviar, calidad aceptable")
print("- Si necesitas guardar millones de imágenes en un servidor")
print(" → Usa 10 componentes: ahorra 50% del espacio, calidad excelente")
print("=" * 60)

### 1.2. Descomposición QR

#### ¿Qué es la Descomposición QR?

La **descomposición QR** es otra forma de descomponer una matriz, pero con un propósito diferente a SVD.

**Analogía del mundo real - Construcción de un edificio:**
Imagina que tienes que construir un edificio con columnas torcidas e irregulares:
- La matriz **Q** es como un conjunto de columnas perfectamente verticales y perpendiculares entre sí
- La matriz **R** contiene las instrucciones de cómo "inclinar" esas columnas perfectas para obtener tu edificio original

La descomposición QR expresa una matriz A como:

$$A = QR$$

Donde:
- **Q:** Matriz **ortogonal** (sus columnas son perpendiculares entre sí y tienen longitud 1)
  - Propiedad importante: Q^T @ Q = I (la matriz identidad)
  - Piensa en Q como vectores perfectamente alineados en un sistema de coordenadas
  
- **R:** Matriz **triangular superior** (todos los valores por debajo de la diagonal son 0)
  - Contiene las "instrucciones" de cómo combinar las columnas de Q
  - Es más fácil de trabajar que una matriz llena de números

#### ¿Por qué usar QR en lugar de otras descomposiciones?

**Razón 1: Estabilidad numérica**
- Invertir una matriz directamente puede dar errores enormes si la matriz está "mal condicionada"
- QR es mucho más estable numéricamente
- Analogía: Es como usar GPS en lugar de brújula en una tormenta

**Razón 2: Resolver sistemas de ecuaciones**
- En machine learning constantemente resolvemos sistemas como: Ax = b
- Ejemplo típico: Regresión lineal
- Con QR es más eficiente y preciso que con inversión directa

**Razón 3: Sistemas sobre-determinados**
- Cuando tienes MÁS ecuaciones que incógnitas (muy común en ML)
- Ejemplo: 1000 puntos de datos pero solo 2 variables (peso y altura)
- QR encuentra la **mejor aproximación** (mínimos cuadrados)

#### Aplicaciones prácticas:

1. **Regresión lineal:**
   - Encontrar la recta que mejor se ajusta a tus datos
   - Más estable que calcular (XᵀX)⁻¹Xᵀy directamente

2. **Cálculo de determinantes y rangos:**
   - El determinante se calcula fácilmente como el producto de la diagonal de R

3. **Algoritmo para calcular eigenvalues:**
   - El famoso "Algoritmo QR" para encontrar valores propios

4. **Resolver ecuaciones en redes neuronales:**
   - Algunas capas requieren resolver sistemas lineales eficientemente

In [ ]:
# ============================================================
# EJEMPLO PRÁCTICO: Resolver sistema de ecuaciones con QR
# ============================================================
#
# PROBLEMA: Encontrar la mejor línea que se ajusta a unos puntos
# (Regresión lineal usando QR)
#
# CONTEXTO: Tenemos más ecuaciones que incógnitas (sistema sobre-determinado)
# No hay solución exacta, buscamos la MEJOR APROXIMACIÓN
# ============================================================

print("=" * 70)
print("PROBLEMA: REGRESIÓN LINEAL CON DESCOMPOSICIÓN QR")
print("=" * 70)
print()

# PASO 1: Crear un sistema sobre-determinado
# --------------------------------------------
# Sistema de ecuaciones: A @ x = b
# donde A es una matriz con MÁS filas que columnas

# Matriz A: representa nuestros datos de entrada
# Cada fila es una observación, cada columna es una variable
A = np.array([
    [1, 2], # Observación 1: x1=1, x2=2
    [3, 4], # Observación 2: x1=3, x2=4
    [5, 6], # Observación 3: x1=5, x2=6
    [7, 8] # Observación 4: x1=7, x2=8
], dtype=float)

# Vector b: los valores que queremos predecir
b = np.array([1, 2, 3, 4], dtype=float)

print("SISTEMA DE ECUACIONES:")
print("-" * 70)
print(f"Matriz A (datos de entrada):")
print(A)
print(f" Shape: {A.shape} → {A.shape[0]} ecuaciones, {A.shape[1]} incógnitas")
print()
print(f"Vector b (valores a predecir):")
print(b)
print(f" Shape: {b.shape}")
print()
print("OBSERVACIÓN: Tenemos 4 ecuaciones pero solo 2 incógnitas")
print(" → Sistema SOBRE-DETERMINADO (no hay solución exacta)")
print(" → Buscamos la MEJOR aproximación (mínimos cuadrados)")
print()

# PASO 2: Aplicar Descomposición QR
# -----------------------------------
print("=" * 70)
print("APLICANDO DESCOMPOSICIÓN QR")
print("=" * 70)
print()

Q, R = np.linalg.qr(A)

print("Resultado de la descomposición:")
print()
print("Matriz Q (ortogonal):")
print(f" Shape: {Q.shape}")
print(Q)
print()
print("Matriz R (triangular superior):")
print(f" Shape: {R.shape}")
print(R)
print()

# PASO 3: Verificar que Q es ortogonal
# --------------------------------------
# Una matriz ortogonal cumple: Q^T @ Q = I (matriz identidad)
print("=" * 70)
print("VERIFICACIÓN: ¿Q es realmente ortogonal?")
print("=" * 70)
print()

identidad = Q.T @ Q
print("Q^T @ Q (debería ser la matriz identidad):")
print(np.round(identidad, decimals=10))
print()

# Verificar que es la identidad
es_identidad = np.allclose(identidad, np.eye(len(identidad)))
print(f"¿Es la matriz identidad? {es_identidad} ✓")
print()
print("EXPLICACIÓN: Si Q^T @ Q = I, significa que las columnas de Q son:")
print(" 1. Perpendiculares entre sí (ortogonales)")
print(" 2. De longitud 1 (normalizadas)")
print(" → Esto hace a Q muy estable numéricamente")
print()

# PASO 4: Resolver el sistema usando QR
# ---------------------------------------
print("=" * 70)
print("RESOLVIENDO EL SISTEMA")
print("=" * 70)
print()

print("Queremos resolver: A @ x = b")
print("Sabemos que: A = Q @ R")
print("Entonces: Q @ R @ x = b")
print()
print("Multiplicamos ambos lados por Q^T:")
print(" Q^T @ Q @ R @ x = Q^T @ b")
print(" Como Q^T @ Q = I:")
print(" R @ x = Q^T @ b")
print()
print("Ahora tenemos un sistema triangular R @ x = Q^T @ b")
print("que es MUY fácil de resolver")
print()

# Calcular Q^T @ b
Qtb = Q.T @ b
print(f"Q^T @ b = {Qtb}")
print()

# Resolver R @ x = Q^T @ b
# Solo usamos las primeras 2 filas de R (las que tienen información)
x = np.linalg.solve(R[:2, :2], Qtb[:2])

print(f"SOLUCIÓN (mejor aproximación): x = {x}")
print()

# PASO 5: Verificar la calidad de la solución
# ---------------------------------------------
print("=" * 70)
print("VERIFICACIÓN DE LA SOLUCIÓN")
print("=" * 70)
print()

# Calcular las predicciones
predicciones = A @ x
print("Comparación de valores reales vs predichos:")
print(f"{'Real':<10} {'Predicho':<10} {'Error':<10}")
print("-" * 30)
for real, pred in zip(b, predicciones):
    error = abs(real - pred)
    print(f"{real:<10.4f} {pred:<10.4f} {error:<10.4f}")

print()
# Calcular el error total (norma L2)
error_total = np.linalg.norm(A @ x - b)
print(f"Error total (norma L2): {error_total:.6f}")
print()
print("INTERPRETACIÓN:")
print(f" - No hay solución exacta (sistema sobre-determinado)")
print(f" - La solución encontrada x = {x} es la MEJOR posible")
print(f" - Minimiza el error cuadrático (mínimos cuadrados)")
print()

# PASO 6: Ventajas de usar QR
# -----------------------------
print("=" * 70)
print("¿POR QUÉ USAR QR EN VEZ DE INVERTIR LA MATRIZ?")
print("=" * 70)
print()
print("Método tradicional (MALO):")
print(" x = (A^T @ A)^(-1) @ A^T @ b")
print(" Problemas:")
print(" - Calcular la inversa es costoso computacionalmente")
print(" - Muy inestable si A^T @ A está mal condicionada")
print(" - Errores numéricos pueden ser ENORMES")
print()
print("Método QR (BUENO):")
print(" 1. Descomponer A = Q @ R")
print(" 2. Resolver R @ x = Q^T @ b (sistema triangular, muy rápido)")
print(" Ventajas:")
print(" ✓ Más rápido (menos operaciones)")
print(" ✓ Numéricamente estable")
print(" ✓ Menos errores de redondeo")
print(" ✓ Funciona bien con matrices mal condicionadas")
print()
print("CONCLUSIÓN: En machine learning, SIEMPRE usa QR (o SVD) en vez de")
print(" invertir matrices directamente")
print("=" * 70)

### 1.3. Valores y Vectores Propios (Eigenvalues y Eigenvectors)
https://www.youtube.com/shorts/HRvnxPOf1Fw
#### ¿Qué son los Eigenvalues y Eigenvectors?

Para una matriz cuadrada A, un **valor propio** λ (lambda) y su **vector propio** v cumplen:

$$Av = \lambda v$$

**¿Qué significa esto en palabras simples?**

Cuando multiplicas una matriz A por un vector v:
- **Normalmente:** El vector cambia de dirección Y de tamaño
- **Vector propio:** El vector SOLO cambia de tamaño, NO de dirección
- **Valor propio:** Es el número que te dice cuánto cambia el tamaño

#### Analogía del mundo real - Estirar una camiseta:

Imagina que estiras una camiseta elástica:
- La mayoría de puntos de la tela se mueven en direcciones complicadas
- Pero HAY direcciones especiales:
  - **Eje horizontal:** Si estiras horizontalmente, los puntos en este eje solo se mueven horizontalmente
  - **Eje vertical:** Si estiras verticalmente, los puntos en este eje solo se mueven verticalmente
  
- Estas **direcciones especiales** son los **eigenvectors** (vectores propios)
- **Cuánto se estiran** en cada dirección son los **eigenvalues** (valores propios)
  - λ = 2 → Se estira el doble
  - λ = 0.5 → Se encoge a la mitad
  - λ = 1 → No cambia de tamaño
  - λ = -1 → Se invierte la dirección pero mantiene el tamaño

#### ¿Por qué son importantes en IA y Data Science?

**1. PCA (Principal Component Analysis) - Reducción de dimensionalidad:**

Imagina que tienes datos con 100 variables (features). Muchas están correlacionadas y son redundantes.

- Los **eigenvectors** de la matriz de covarianza te dicen **en qué direcciones hay más variación** en tus datos
- Los **eigenvalues** te dicen **cuánta información** hay en cada dirección
- El eigenvector con el eigenvalue MÁS GRANDE es el **componente principal** (PC1)
- Puedes quedarte solo con los primeros componentes y descartar el resto

**Ejemplo práctico:**
```
100 variables originales → PCA → 10 componentes principales
- Conservas el 95% de la información
- Reducción de 100 a 10 dimensiones
- Los modelos entrenan 10x más rápido
- Reduces overfitting
```

**2. Google PageRank:**
- Cada página web es un nodo en un grafo gigante
- Los enlaces entre páginas forman una matriz A
- El **eigenvector principal** te dice qué páginas son más importantes
- Google usa esto para ordenar los resultados de búsqueda

**3. Sistemas dinámicos y estabilidad:**
- En redes neuronales recurrentes (RNN), necesitas saber si el sistema es estable
- Los eigenvalues te dicen si el sistema explota (|λ| > 1) o converge (|λ| < 1)

**4. Análisis de grafos y redes sociales:**
- Detectar comunidades en redes
- Identificar usuarios influyentes
- Clustering espectral (usar eigenvalues para agrupar datos)

#### Interpretación geométrica:

Cuando aplicas una transformación matricial A a un espacio:
- Los **eigenvectors** son las direcciones que NO se tuercen, solo se estiran/encogen
- Los **eigenvalues** indican cuánto se estiran/encogen esas direcciones

**Ejemplo visual:**
```
Si A tiene eigenvalues: λ₁ = 3, λ₂ = 0.5

Significa que:
- En la dirección del eigenvector v₁: todo se triplica (×3)
- En la dirección del eigenvector v₂: todo se reduce a la mitad (×0.5)
```

#### Propiedades importantes:

1. **Matriz simétrica** → Eigenvalues reales, eigenvectors perpendiculares
   - Muy común en datos reales (matrices de covarianza)
   
2. **Suma de eigenvalues** = Suma de elementos de la diagonal (traza de la matriz)

3. **Producto de eigenvalues** = Determinante de la matriz
   - Si algún eigenvalue es 0 → La matriz no es invertible

4. **Eigenvalues negativos** → La transformación invierte dirección en ese eje

In [ ]:
# ============================================================
# EJEMPLO PRÁCTICO: PCA simplificado usando eigenvalues
# ============================================================
#
# OBJETIVO:
# Encontrar las direcciones de máxima varianza en un
# conjunto de datos 2D y decidir si podemos reducir
# dimensionalidad sin perder demasiada información.
#
# PASOS:
# 1. Generar datos 2D correlacionados
# 2. Centrar los datos (restar media)
# 3. Calcular matriz de covarianza
# 4. Calcular eigenvalues/eigenvectors
# 5. Ordenarlos por importancia
# 6. Interpretar resultados
# ============================================================

import numpy as np

print("=" * 70)
print("PCA (PRINCIPAL COMPONENT ANALYSIS) CON EIGENVALUES")
print("=" * 70)
print()

# PASO 1: Generar datos 2D correlacionados
rng = np.random.default_rng(seed=42)
n = 100

x = rng.normal(0, 2, n)
y = 1.5*x + rng.normal(0, 0.5, n)

datos = np.vstack([x, y]).T

print("DATOS ORIGINALES:")
print("-" * 70)
print(f"Shape: {datos.shape} → {datos.shape[0]} puntos, {datos.shape[1]} dimensiones")
print("\nPrimeras 5 muestras:")
print(f"{'X':<12} {'Y':<12}")
print("-"*24)
for i in range(5):
    print(f"{datos[i,0]:<12.4f} {datos[i,1]:<12.4f}")
print()

print("Estadísticas:")
print(f" Media X: {np.mean(datos[:,0]):.4f}")
print(f" Media Y: {np.mean(datos[:,1]):.4f}")
print(f" Std X: {np.std(datos[:,0]):.4f}")
print(f" Std Y: {np.std(datos[:,1]):.4f}")
print()

# PASO 2: Centrado
print("=" * 70)
print("CENTRANDO LOS DATOS")
print("=" * 70)
print()

media = np.mean(datos, axis=0)
datos_c = datos - media

print(f"Media original: {media}")
print(f"Nueva media después de centrar: {np.mean(datos_c, axis=0)}")
print("→ Datos centrados correctamente\n")

# PASO 3: Matriz de covarianza
print("=" * 70)
print("MATRIZ DE COVARIANZA")
print("=" * 70)
print()

cov = np.cov(datos_c.T)
print("Matriz de covarianza:")
print(cov)
print()

print("Interpretación:")
print(f" Var(X) = cov[0,0] = {cov[0,0]:.4f}")
print(f" Var(Y) = cov[1,1] = {cov[1,1]:.4f}")
print(f" Cov(X,Y) = cov[0,1] = {cov[0,1]:.4f}")
print("→ Covarianza positiva → Las variables están correlacionadas\n")

# PASO 4: Eigenvalues y eigenvectors
print("=" * 70)
print("EIGENVALUES Y EIGENVECTORS")
print("=" * 70)
print()

eig_vals, eig_vecs = np.linalg.eigh(cov)

# PASO 5: Ordenar de mayor a menor (CRUCIAL)
idx = np.argsort(eig_vals)[::-1]
eig_vals = eig_vals[idx]
eig_vecs = eig_vecs[:, idx]

print("Eigenvalues ordenados:")
print(f" λ1 = {eig_vals[0]:.4f} (máxima varianza)")
print(f" λ2 = {eig_vals[1]:.4f} (mínima varianza)\n")

total_var = np.sum(eig_vals)
ratio = eig_vals / total_var * 100

print("Varianza explicada por componente:")
print(f" PC1: {ratio[0]:.2f}%")
print(f" PC2: {ratio[1]:.2f}%\n")

print("Eigenvectors (direcciones principales):")
print(eig_vecs)
print()
print("Interpretación:")
print(f" - PC1: {eig_vecs[:,0]} → Captura el {ratio[0]:.1f}% de la varianza (dirección principal)")
print(f" - PC2: {eig_vecs[:,1]} → Captura el {ratio[1]:.1f}% (dirección perpendicular)\n")

# PASO 6: Decidir reducción dimensional
print("=" * 70)
print("DECISIÓN: ¿REDUCIMOS DIMENSIONALIDAD?")
print("=" * 70)

print("\nSi queremos conservar el 90% de la información:")
if ratio[0] >= 90:
    print(f"→ Basta con PC1 ({ratio[0]:.2f}%)")
    print("→ Podemos reducir de 2D a 1D casi sin perder información\n")
else:
    print("→ Se necesitan las 2 dimensiones\n")

print("Acumulado de varianza:")
print(f" 1 componente: {ratio[0]:.2f}%")
print(f" 2 componentes: {ratio[0] + ratio[1]:.2f}%\n")

print("=" * 70)
print("CONCLUSIÓN FINAL")
print("=" * 70)

print(f"- PC1 captura {ratio[0]:.1f}% de la información → componente útil")
print(f"- PC2 captura solo {ratio[1]:.1f}% → ruido o variación menor")
print("- Podemos reducir el dataset de 2D a 1D usando solo PC1")
print("=" * 70)


In [ ]:
# ============================================================
# VISUALIZACIÓN DEL PCA
# ============================================================

import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

# ------------------------------------------------------------
# Subplot 1: Datos originales centrados + componentes principales
# ------------------------------------------------------------
plt.subplot(1, 2, 1)
plt.scatter(datos_c[:, 0], datos_c[:, 1], alpha=0.6, s=30)

escala = 3
for i in range(2):
    # Dibujar flecha de cada componente principal (PC1 y PC2)
    plt.arrow(0, 0,
              eig_vecs[0, i] * eig_vals[i] * escala,
              eig_vecs[1, i] * eig_vals[i] * escala,
              head_width=0.2, head_length=0.2,
              fc=f'C{i}', ec=f'C{i}', linewidth=2)

    # Identificar cada componente
    plt.text(eig_vecs[0, i] * eig_vals[i] * escala * 1.2,
             eig_vecs[1, i] * eig_vals[i] * escala * 1.2,
             f'PC{i+1}',
             fontsize=12, fontweight='bold')

plt.xlabel('Feature 1 (centrado)')
plt.ylabel('Feature 2 (centrado)')
plt.title('Datos Originales + Componentes Principales')
plt.grid(True, alpha=0.3)
plt.axis('equal')

# ------------------------------------------------------------
# Subplot 2: Datos proyectados sobre los componentes principales
# ------------------------------------------------------------
datos_pca = datos_c @ eig_vecs # proyección en PC1 y PC2

plt.subplot(1, 2, 2)
plt.scatter(datos_pca[:, 0], datos_pca[:, 1], alpha=0.6, s=30, c='green')

plt.xlabel('PC1 (mayor varianza)')
plt.ylabel('PC2 (menor varianza)')
plt.title('Datos Proyectados en Componentes Principales')
plt.grid(True, alpha=0.3)
plt.axis('equal')

plt.tight_layout()
plt.show()

print("Observa cómo PC1 captura claramente la mayor parte de la varianza.")
# ============================================================
# GRÁFICO 3: REDUCCIÓN DE 2D A 1D USANDO SOLO PC1
# ============================================================

plt.figure(figsize=(12, 5))

# --------------------------
# Subplot A: Datos en 2D
# --------------------------
plt.subplot(1, 2, 1)
plt.scatter(datos_c[:, 0], datos_c[:, 1], alpha=0.6, s=30)
plt.title("Datos Originales (2D)")
plt.xlabel("X centrado")
plt.ylabel("Y centrado")
plt.grid(True, alpha=0.3)
plt.axis("equal")

# Dibujar PC1 como línea
pc1_vector = eig_vecs[:, 0] * 3
plt.arrow(0, 0, pc1_vector[0], pc1_vector[1],
          head_width=0.2, head_length=0.2, color="red")

plt.text(pc1_vector[0] * 1.1,
         pc1_vector[1] * 1.1,
         "PC1", fontsize=12, fontweight="bold", color="red")


# --------------------------
# Subplot B: Datos proyectados en PC1 (1D)
# --------------------------
proyeccion_pc1 = datos_c @ eig_vecs[:, 0] # solo componente principal

plt.subplot(1, 2, 2)
plt.scatter(proyeccion_pc1, np.zeros_like(proyeccion_pc1),
            alpha=0.6, s=30, c='purple')

plt.title("Datos Reducidos a 1D usando PC1")
plt.xlabel("PC1 (1 dimensión)")
plt.yticks([]) # quitar eje Y porque ya no existe
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Aquí puedes ver claramente cómo los datos pasan de un plano 2D a una línea 1D usando solo PC1.")



### 1.4. Normas de Vectores y Matrices

Las **normas** miden el "tamaño" o "magnitud" de vectores y matrices. Son fundamentales en:

- **Regularización en ML:** L1 (Lasso), L2 (Ridge)
- **Optimización:** Medir convergencia de gradientes
- **Normalización de embeddings:** En NLP y sistemas de recomendación

**Normas comunes:**
- **L1 (Manhattan):** Suma de valores absolutos, `np.linalg.norm(v, ord=1)`
- **L2 (Euclidiana):** Raíz de suma de cuadrados, `np.linalg.norm(v, ord=2)` o `np.linalg.norm(v)`
- **L∞ (Infinito):** Valor absoluto máximo, `np.linalg.norm(v, ord=np.inf)`

In [ ]:
# Ejemplo: Diferentes normas de vectores

vector = np.array([3, -4, 0, 5])

print(f"Vector: {vector}")
print()

# Norma L1 (Manhattan)
norma_l1 = np.linalg.norm(vector, ord=1)
print(f"Norma L1 (Manhattan): {norma_l1}")
print(f" Cálculo: |3| + |-4| + |0| + |5| = {abs(3) + abs(-4) + abs(0) + abs(5)}")
print()

# Norma L2 (Euclidiana)
norma_l2 = np.linalg.norm(vector, ord=2) # o simplemente np.linalg.norm(vector)
print(f"Norma L2 (Euclidiana): {norma_l2:.4f}")
print(f" Cálculo: √(3² + (-4)² + 0² + 5²) = √{3**2 + 4**2 + 0**2 + 5**2} = {np.sqrt(50):.4f}")
print()

# Norma L∞ (Infinito)
norma_linf = np.linalg.norm(vector, ord=np.inf)
print(f"Norma L∞ (Infinito): {norma_linf}")
print(f" Cálculo: max(|3|, |-4|, |0|, |5|) = {max(abs(3), abs(-4), abs(0), abs(5))}")
print()

# Normalización L2 (común en ML)
vector_normalizado = vector / norma_l2
print(f"Vector normalizado L2: {vector_normalizado}")
print(f"Norma del vector normalizado: {np.linalg.norm(vector_normalizado):.4f} (debe ser 1.0)")

## 2. Operaciones Avanzadas con Arrays

### 2.1. Views vs Copies: Gestión de Memoria

#### ¿Qué es una View (Vista) y qué es una Copy (Copia)?

Uno de los conceptos MÁS IMPORTANTES para optimizar código NumPy es entender la diferencia entre **views** y **copies**.

**Analogía del mundo real - Documento de Google Docs:**

**COPY (Copia) - Documento duplicado:**
- Haces una copia completa del documento
- Ahora tienes 2 archivos independientes
- Si editas la copia, el original NO cambia
- Ocupa EL DOBLE de espacio en tu Google Drive
- **En NumPy:** Se crea un nuevo bloque de memoria con los mismos datos

**VIEW (Vista) - Compartir link del documento:**
- Compartes un enlace al mismo documento
- Sigue siendo UN SOLO archivo
- Si alguien edita a través del enlace, el original CAMBIA
- No ocupa espacio adicional (es el mismo archivo)
- **En NumPy:** Se crea una referencia al mismo bloque de memoria

#### ¿Por qué es CRÍTICO entender esto?

**Problema 1: Bugs sutiles y difíciles de detectar**
```python
# Piensas que haces una copia, pero es una vista
a = np.array([1, 2, 3, 4, 5])
b = a[1:4] # ¡Esto es una VIEW, no una copia!
b[0] = 999 # Modificas b...
# ¡Sorpresa! También modificaste a
```

**Problema 2: Desperdicio de memoria**
```python
# Si copias un array de 1GB sin necesidad
big_array = np.random.rand(1000, 1000, 100) # ~800 MB
copia_innecesaria = big_array.copy() # Otros 800 MB
# ¡Ahora usas 1.6 GB en vez de 800 MB!
```

**Problema 3: Lentitud innecesaria**
```python
# Copiar arrays grandes es COSTOSO
# Si solo necesitas leer datos, usa views (gratis)
# Si necesitas modificar sin afectar el original, entonces sí copia
```

#### Reglas de oro: ¿Cuándo se crea View vs Copy?

| Operación | Resultado | Ejemplo |
|-----------|-----------|---------|
| **Slicing básico** | VIEW | `a[1:5]`, `a[::2]`, `a[:, 1]` |
| **Fancy indexing** (array de índices) | COPY | `a[[1, 3, 5]]` |
| **Boolean indexing** | COPY | `a[a > 5]` |
| **`.reshape()`** | VIEW (si es posible) | `a.reshape(2, 3)` |
| **`.ravel()`** | VIEW (si es posible) | `a.ravel()` |
| **`.flatten()`** | COPY (siempre) | `a.flatten()` |
| **`.T` (transpuesta)** | VIEW | `a.T` |
| **`.copy()`** | COPY (explícita) | `a.copy()` |
| **Operaciones aritméticas** | COPY (nuevo array) | `a + 5`, `a * 2` |

#### ¿Cómo verificar si es View o Copy?

**Método 1: Atributo `.base`**
```python
if arr.base is not None:
    print("Es una VIEW")
else:
    print("Es una COPY (o array original)")
```

**Método 2: Modificar y ver si afecta al original**
```python
original = np.array([1, 2, 3])
resultado = original[:]
resultado[0] = 999
# Si original cambió → VIEW
# Si original NO cambió → COPY
```

#### ¿Cuándo usar cada una?

**Usa VIEWS cuando:**
- ✓ Solo necesitas LEER datos (no modificar)
- ✓ Quieres ahorrar memoria (arrays grandes)
- ✓ Necesitas trabajar con subarrays temporalmente
- ✓ El rendimiento es crítico

**Usa COPIES cuando:**
- ✓ Vas a MODIFICAR los datos y NO quieres afectar el original
- ✓ Necesitas preservar el estado original
- ✓ Pasas arrays a funciones que los modifican in-place
- ✓ No estás seguro → mejor copia explícita

#### Mejores prácticas:

1. **Siempre usa `.copy()` cuando tengas dudas**
   ```python
   # MALO (arriesgado):
   subset = datos[mask] # ¿View o copy? Depende de mask
   
   # BUENO (seguro):
   subset = datos[mask].copy() # Siempre es copia
   ```

2. **Documenta si una función modifica el array original**
   ```python
   def procesar(arr: np.ndarray, inplace: bool = False) -> np.ndarray:
       """
       Procesa el array.
       
       Args:
           arr: Array de entrada
           inplace: Si True, modifica arr directamente (más rápido)
                   Si False, trabaja en una copia (más seguro)
       """
       if not inplace:
           arr = arr.copy()
       # ... procesamiento ...
       return arr
   ```

3. **Verifica antes de modificar**
   ```python
   if resultado.base is not None:
       print(" CUIDADO: Es una view, modificar afectará al original")
       resultado = resultado.copy()
   ```

In [ ]:
# ============================================================
# EJEMPLO 1: Slicing crea una VIEW
# ============================================================
#
# CONCEPTO: El slicing básico (usando :) NO copia los datos
# Solo crea una "ventana" al array original
#
# CONSECUENCIA: Si modificas la vista, modificas el original
# ============================================================

print("=" * 70)
print("EJEMPLO 1: SLICING → VIEW")
print("=" * 70)
print()

# Crear array original
original = np.array([1, 2, 3, 4, 5])
print("PASO 1: Crear array original")
print(f" original = {original}")
print()

# Crear vista con slicing (extraer elementos 1, 2, 3)
print("PASO 2: Crear 'vista' con slicing")
vista = original[1:4] # Posiciones 1, 2, 3 → valores [2, 3, 4]
print(f" vista = original[1:4]")
print(f" vista = {vista}")
print()

# Verificar que es una vista
print("PASO 3: Verificar si es VIEW o COPY")
if vista.base is original:
    print(f" ✓ Es una VIEW (vista.base is original = True)")
    print(f" → vista y original comparten la MISMA memoria")
else:
    print(f" ✗ Es una COPY")
print()

# Mostrar IDs de memoria (solo informativo)
print(" Comprobación adicional:")
print(f" ID del original: {id(original)}")
print(f" ID de la base de vista: {id(vista.base)}")
print(f" ¿Son el mismo objeto? {vista.base is original}")
print()

# EXPERIMENTO: Modificar la vista
print("=" * 70)
print("EXPERIMENTO: ¿Qué pasa si modifico la vista?")
print("=" * 70)
print()

print("ANTES de modificar:")
print(f" original = {original}")
print(f" vista = {vista}")
print()

# Modificar el primer elemento de la vista
print("Ejecutando: vista[0] = 999")
vista[0] = 999
print()

print("DESPUÉS de modificar:")
print(f" original = {original} ← ¡¡¡CAMBIÓ!!!")
print(f" vista = {vista}")
print()

print("EXPLICACIÓN:")
print(" - vista[0] corresponde a original[1] (segundo elemento)")
print(" - Al cambiar vista[0] a 999, cambiamos original[1] a 999")
print(" - Porque vista NO es una copia, es una VENTANA al original")
print()

print("=" * 70)
print("DIAGRAMA CONCEPTUAL")
print("=" * 70)
print()
print("MEMORIA:")
print(" ┌───┬───┬───┬───┬───┐")
print(" │ 1 │999│ 3 │ 4 │ 5 │ ← Array original en memoria")
print(" └───┴───┴───┴───┴───┘")
print(" ↑ ↑ ↑")
print(" └───┴───┘")
print(" vista[0:3] ← Vista apunta a la misma memoria")
print()

print("CONCLUSIÓN:")
print(" Slicing crea una VIEW, NO una copia")
print(" Modificar la vista modifica el original")
print(" ✓ Si necesitas independencia, usa .copy()")
print("=" * 70)

In [ ]:
# ============================================================
# EJEMPLO 2: Fancy Indexing crea una COPY
# ============================================================
#
# CONCEPTO: Cuando usas un ARRAY de índices (fancy indexing),
# NumPy CREA UNA COPIA de los datos
#
# RAZÓN: Los elementos pueden no estar contiguos en memoria,
# así que NumPy necesita copiarlos a un nuevo array
#
# CONSECUENCIA: Modificar el resultado NO afecta al original
# ============================================================

print("=" * 70)
print("EJEMPLO 2: FANCY INDEXING → COPY")
print("=" * 70)
print()

# Crear array original
original = np.array([1, 2, 3, 4, 5])
print("PASO 1: Crear array original")
print(f" original = {original}")
print()

# Fancy indexing: seleccionar elementos con un array de índices
print("PASO 2: Usar fancy indexing (array de índices)")
indices = np.array([1, 2, 3])
print(f" indices = {indices}")
copia = original[indices] # Selecciona original[1], original[2], original[3]
print(f" copia = original[indices]")
print(f" copia = {copia}")
print()

print("EXPLICACIÓN:")
print(" - original[indices] selecciona elementos en posiciones [1, 2, 3]")
print(" - Esto extrae valores [2, 3, 4]")
print(" - IMPORTANTE: Usa un ARRAY de índices (no slicing)")
print()

# Verificar que es una copia
print("PASO 3: Verificar si es VIEW o COPY")
print(f" copia.base es None: {copia.base is None}")
if copia.base is None:
    print(f" ✓ Es una COPY (copia.base = None)")
    print(f" → copia y original tienen memoria SEPARADA")
else:
    print(f" ✗ Es una VIEW")
print()

# Mostrar que son objetos diferentes
print(" Comprobación adicional:")
print(f" ID del original: {id(original)}")
print(f" ID de la copia: {id(copia)}")
print(f" ¿Son el mismo objeto? {copia is original}")
print()

# EXPERIMENTO: Modificar la copia
print("=" * 70)
print("EXPERIMENTO: ¿Qué pasa si modifico la copia?")
print("=" * 70)
print()

print("ANTES de modificar:")
print(f" original = {original}")
print(f" copia = {copia}")
print()

# Modificar el primer elemento de la copia
print("Ejecutando: copia[0] = 999")
copia[0] = 999
print()

print("DESPUÉS de modificar:")
print(f" original = {original} ← NO cambió ✓")
print(f" copia = {copia}")
print()

print("EXPLICACIÓN:")
print(" - copia[0] se cambia a 999")
print(" - Pero original[1] sigue siendo 2 (no cambia)")
print(" - Porque copia es un array INDEPENDIENTE con su propia memoria")
print()

print("=" * 70)
print("DIAGRAMA CONCEPTUAL")
print("=" * 70)
print()
print("MEMORIA:")
print(" Array original:")
print(" ┌───┬───┬───┬───┬───┐")
print(" │ 1 │ 2 │ 3 │ 4 │ 5 │ ← Memoria del original")
print(" └───┴───┴───┴───┴───┘")
print()
print(" Array copia (SEPARADO):")
print(" ┌───┬───┬───┐")
print(" │999│ 3 │ 4 │ ← Memoria de la copia (independiente)")
print(" └───┴───┴───┘")
print()

print("=" * 70)
print("COMPARACIÓN: SLICING vs FANCY INDEXING")
print("=" * 70)
print()
print(" original[1:4] → VIEW (slicing con :)")
print(" original[[1, 2, 3]] → COPY (fancy indexing con array)")
print()
print("¿Cuándo usar cada uno?")
print()
print(" Usa SLICING cuando:")
print(" ✓ Quieres elementos consecutivos")
print(" ✓ Quieres ahorrar memoria")
print(" ✓ Solo vas a LEER (no modificar)")
print()
print(" Usa FANCY INDEXING cuando:")
print(" ✓ Necesitas elementos NO consecutivos")
print(" ✓ Vas a modificar y NO quieres afectar el original")
print(" ✓ Necesitas reordenar elementos")
print()

print("CONCLUSIÓN:")
print(" Fancy indexing SIEMPRE crea una COPY")
print(" ✓ Modificar la copia NO afecta al original")
print(" ✓ Útil cuando necesitas independencia de datos")
print("=" * 70)

In [ ]:
# ============================================================
# EJEMPLO 3: Crear copias EXPLÍCITAS con .copy()
# ============================================================
#
# CONCEPTO: Cuando tienes dudas, usa .copy() para garantizar
# que tienes un array independiente
#
# REGLA DE ORO: Si vas a modificar Y no quieres afectar el
# original → USA .copy()
# ============================================================

print("=" * 70)
print("EJEMPLO 3: COPIAS EXPLÍCITAS CON .copy()")
print("=" * 70)
print()

# Crear array original
original = np.array([1, 2, 3, 4, 5])
print("PASO 1: Crear array original")
print(f" original = {original}")
print()

# Opción MALA: Slicing (es una vista)
print("OPCIÓN 1 (NO SEGURA): Slicing normal")
vista_arriesgada = original[:3] # Primeros 3 elementos
print(f" vista_arriesgada = original[:3]")
print(f" Resultado: {vista_arriesgada}")
print(f" ¿Es vista? {vista_arriesgada.base is not None}")
print(" Si modificas esto, cambiarás el original")
print()

# Opción BUENA: Slicing + .copy()
print("OPCIÓN 2 (SEGURA): Slicing + .copy()")
copia_segura = original[:3].copy() # Primeros 3 elementos COPIADOS
print(f" copia_segura = original[:3].copy()")
print(f" Resultado: {copia_segura}")
print(f" ¿Es copia? {copia_segura.base is None}")
print(" ✓ Puedes modificar sin miedo, el original no cambiará")
print()

# DEMOSTRACIÓN práctica
print("=" * 70)
print("DEMOSTRACIÓN: Modificar ambas y ver diferencia")
print("=" * 70)
print()

print("Estado inicial:")
print(f" original = {original}")
print(f" vista_arriesgada = {vista_arriesgada}")
print(f" copia_segura = {copia_segura}")
print()

print("Modificando ambas: ponemos 999 en la primera posición")
vista_arriesgada[0] = 999
copia_segura[0] = 888
print()

print("Después de modificar:")
print(f" original = {original} ← ¡Cambió a 999! (por la vista)")
print(f" vista_arriesgada = {vista_arriesgada}")
print(f" copia_segura = {copia_segura} ← Original NO afectado")
print()

print("OBSERVA:")
print(" - vista_arriesgada[0] = 999 → también cambió original[0] a 999")
print(" - copia_segura[0] = 888 → NO afectó al original")
print()

# Ejemplo real: función que modifica datos
print("=" * 70)
print("CASO PRÁCTICO: Función que procesa datos")
print("=" * 70)
print()

def normalizar_sin_copy(arr):
    """PELIGROSO: Modifica el array original."""
    arr -= np.mean(arr) # Restar media (modifica in-place)
    arr /= np.std(arr) # Dividir por std (modifica in-place)
    return arr

def normalizar_con_copy(arr):
    """SEGURO: Trabaja en una copia."""
    arr_copy = arr.copy()
    arr_copy -= np.mean(arr_copy)
    arr_copy /= np.std(arr_copy)
    return arr_copy

# Datos de prueba
datos1 = np.array([10, 20, 30, 40, 50], dtype=float)
datos2 = np.array([10, 20, 30, 40, 50], dtype=float)

print("Datos originales:")
print(f" datos1 = {datos1}")
print(f" datos2 = {datos2}")
print()

print("Llamando a normalizar_sin_copy(datos1)...")
resultado1 = normalizar_sin_copy(datos1)
print(f" Resultado: {resultado1}")
print(f" datos1 ahora: {datos1} ← ¡SE MODIFICÓ!")
print()

print("Llamando a normalizar_con_copy(datos2)...")
resultado2 = normalizar_con_copy(datos2)
print(f" Resultado: {resultado2}")
print(f" datos2 ahora: {datos2} ← NO se modificó ✓")
print()

print("=" * 70)
print("MEJORES PRÁCTICAS")
print("=" * 70)
print()
print("1. CUANDO LEER DATOS (sin modificar):")
print(" → Usa views (sin .copy()) para ahorrar memoria")
print(" subset = datos[mask]")
print()
print("2. CUANDO MODIFICAR DATOS:")
print(" → USA .copy() para proteger el original")
print(" subset = datos[mask].copy()")
print(" subset[0] = nuevo_valor # Seguro")
print()
print("3. EN FUNCIONES:")
print(" → Documenta si modificas el array original")
print(" → Ofrece parámetro 'inplace' para dar opción al usuario")
print()
print("4. CUANDO TENGAS DUDAS:")
print(" → SIEMPRE usa .copy()")
print(" → Mejor usar algo más de memoria que tener bugs")
print()
print("RECUERDA:")
print(" Bugs por views inesperadas son DIFÍCILES de encontrar")
print(" Un .copy() extra es BARATO comparado con horas de debugging")
print("=" * 70)

In [ ]:
# Ejemplo 4: reshape() y transpose() crean views

original = np.arange(12)
print(f"Array original (1D): {original}")

# Reshape a 2D (crea vista si es posible)
matriz = original.reshape(3, 4)
print(f"\nMatriz (reshape 3x4):\n{matriz}")

# Modificar la matriz
matriz[0, 0] = 999
print(f"\nDespués de modificar matriz[0, 0] = 999:")
print(f"Matriz:\n{matriz}")
print(f"Original 1D: {original} ← CAMBIÓ (reshape crea view)")
print()

# Transpuesta también es una vista
traspuesta = matriz.T
print(f"¿Transpuesta es vista? {traspuesta.base is not None}")

### 2.2. Memory Layout: Row-major (C) vs Column-major (Fortran)

#### ¿Qué es el Memory Layout?

Las matrices multidimensionales se almacenan en memoria RAM como una **secuencia lineal de números** (1D). Hay dos formas de organizar una matriz 2D en memoria:

**Analogía del mundo real - Libros en una estantería:**

Imagina que tienes una matriz 3×4 (3 filas, 4 columnas) de libros:

```
Fila 1: [A B C D]
Fila 2: [E F G H]
Fila 3: [I J K L]
```

**ROW-MAJOR (C-order, default en NumPy):**
- Guardas fila por fila en la estantería
- Orden en memoria: `A B C D | E F G H | I J K L`
- Análogo a leer un libro: izquierda→derecha, luego siguiente línea
- Acceder a elementos de la **misma fila** es MÁS RÁPIDO

**COLUMN-MAJOR (Fortran-order, usado en MATLAB y R):**
- Guardas columna por columna en la estantería
- Orden en memoria: `A E I | B F J | C G K | D H L`
- Análogo a leer periódico en columnas: arriba→abajo, luego siguiente columna
- Acceder a elementos de la **misma columna** es MÁS RÁPIDO

#### ¿Por qué importa el orden de memoria?

**1. VELOCIDAD (Cache locality):**

Los procesadores modernos cargan datos en "bloques" (cache lines). Si tus datos están juntos en memoria, todo va mucho más rápido.

```python
# Ejemplo: Procesar filas en array C-order (rápido)
for i in range(n_filas):
    suma = np.sum(array_c[i, :]) # Los elementos están juntos en memoria
    
# Ejemplo: Procesar columnas en array C-order (lento)
for j in range(n_cols):
    suma = np.sum(array_c[:, j]) # Los elementos están separados en memoria
```

**Analogía:**
- Imagina que tienes que recoger 100 manzanas
- **Caso A:** Todas en una caja (memoria contigua) → 1 segundo
- **Caso B:** Cada manzana en una caja diferente (memoria dispersa) → 10 segundos
- ¡Diferencia de 10x solo por cómo están organizadas!

**2. INTEROPERABILIDAD:**
- **NumPy, TensorFlow, PyTorch:** Usan C-order por defecto
- **MATLAB, R, Fortran:** Usan Fortran-order
- Si pasas datos entre librerías, puede haber conversiones automáticas (lentas)

**3. OPERACIONES:**
- **Array C-order:** Iterar por filas es rápido, por columnas es lento
- **Array F-order:** Iterar por columnas es rápido, por filas es lento

#### ¿Cómo saber el orden de un array?

```python
# Verificar flags
print(array.flags['C_CONTIGUOUS']) # True = C-order
print(array.flags['F_CONTIGUOUS']) # True = Fortran-order
```

#### ¿Cuándo usar cada uno?

**Usa C-order (default) cuando:**
- ✓ Trabajas principalmente con filas (ej: procesar imágenes línea por línea)
- ✓ Compatibilidad con la mayoría de librerías Python
- ✓ No tienes una razón específica para cambiarlo

**Usa Fortran-order cuando:**
- ✓ Trabajas principalmente con columnas (ej: operaciones por variables en dataset)
- ✓ Interoperabilidad con MATLAB/R/Fortran
- ✓ Álgebra lineal intensiva (algunas librerías optimizan para F-order)

#### Cómo convertir entre órdenes:

```python
# Crear array en C-order (default)
array_c = np.array([[1, 2], [3, 4]])

# Convertir a Fortran-order
array_f = np.asfortranarray(array_c)

# Convertir a C-order
array_c2 = np.ascontiguousarray(array_f)
```

**IMPORTANTE:** La conversión **copia** los datos (no es gratis).

In [ ]:
# ============================================================
# EXPERIMENTO: Comparar rendimiento C-order vs F-order
# ============================================================
#
# OBJETIVO: Demostrar que el orden de memoria SÍ afecta al
# rendimiento en operaciones reales
#
# EXPERIMENTO:
# - Crear dos arrays idénticos pero con diferente orden
# - Sumar por FILAS → C-order debería ser más rápido
# - Sumar por COLUMNAS → F-order debería ser más rápido
# ============================================================

print("=" * 70)
print("EXPERIMENTO: RENDIMIENTO C-ORDER vs F-ORDER")
print("=" * 70)
print()

# PASO 1: Crear arrays grandes
# ------------------------------
size = 5000 # Matriz 5000×5000 (¡25 millones de elementos!)

print("PASO 1: Crear arrays de prueba")
print(f" Tamaño: {size}×{size} = {size*size:,} elementos")
print()

# Array en C-order (row-major, por defecto en NumPy)
print(" Creando array en C-order (row-major)...")
array_c = np.random.rand(size, size)

# Array en F-order (column-major, como en MATLAB)
print(" Creando array en F-order (column-major)...")
array_f = np.asfortranarray(np.random.rand(size, size))
print()

# Warm-up para evitar overhead inicial
np.sum(array_c, axis=1)
np.sum(array_f, axis=1)

# PASO 2: Verificar el orden de memoria
# ---------------------------------------
print("=" * 70)
print("PASO 2: Verificar flags de memoria")
print("=" * 70)
print()

print("Array C-order:")
print(f" C_CONTIGUOUS: {array_c.flags['C_CONTIGUOUS']}")
print(f" F_CONTIGUOUS: {array_c.flags['F_CONTIGUOUS']}")
print(" → Los datos están organizados por FILAS en memoria")
print()

print("Array F-order:")
print(f" C_CONTIGUOUS: {array_f.flags['C_CONTIGUOUS']}")
print(f" F_CONTIGUOUS: {array_f.flags['F_CONTIGUOUS']}")
print(" → Los datos están organizados por COLUMNAS en memoria")
print()

# PASO 3: Benchmark - Suma por FILAS
# ------------------------------------
print("=" * 70)
print("EXPERIMENTO 1: SUMAR POR FILAS (axis=1)")
print("=" * 70)
print()

print("Operación: np.sum(array, axis=1)")
print(" → Para cada fila, suma todos sus elementos")
print(" → Esperamos que C-order sea MÁS RÁPIDO")
print()

# C-order sumando por filas
inicio = time.perf_counter()
suma_c_filas = np.sum(array_c, axis=1)
tiempo_c_filas = time.perf_counter() - inicio

# F-order sumando por filas
inicio = time.perf_counter()
suma_f_filas = np.sum(array_f, axis=1)
tiempo_f_filas = time.perf_counter() - inicio

print("Resultados:")
print(f" C-order: {tiempo_c_filas:.4f} segundos")
print(f" F-order: {tiempo_f_filas:.4f} segundos")
print()

# Calcular cuál fue más rápido
if tiempo_c_filas < tiempo_f_filas:
    speedup = tiempo_f_filas / tiempo_c_filas
    print(f" ✓ C-order fue {speedup:.2f}x MÁS RÁPIDO")
    print(f" → Porque los elementos de cada fila están JUNTOS en memoria")
else:
    speedup = tiempo_c_filas / tiempo_f_filas
    print(f" F-order fue {speedup:.2f}x más rápido (inesperado)")
print()

print("EXPLICACIÓN:")
print(" Array C-order en memoria: [fila1][fila2][fila3]...")
print(" → Sumar fila1 accede elementos contiguos: RÁPIDO")
print()
print(" Array F-order en memoria: [col1][col2][col3]...")
print(" → Sumar fila1 accede elementos DISPERSOS: LENTO")
print()

# PASO 4: Benchmark - Suma por COLUMNAS
# ---------------------------------------
print("=" * 70)
print("EXPERIMENTO 2: SUMAR POR COLUMNAS (axis=0)")
print("=" * 70)
print()

print("Operación: np.sum(array, axis=0)")
print(" → Para cada columna, suma todos sus elementos")
print(" → Esperamos que F-order sea MÁS RÁPIDO")
print()

# C-order sumando por columnas
inicio = time.perf_counter()
suma_c_cols = np.sum(array_c, axis=0)
tiempo_c_cols = time.perf_counter() - inicio

# F-order sumando por columnas
inicio = time.perf_counter()
suma_f_cols = np.sum(array_f, axis=0)
tiempo_f_cols = time.perf_counter() - inicio

print("Resultados:")
print(f" C-order: {tiempo_c_cols:.4f} segundos")
print(f" F-order: {tiempo_f_cols:.4f} segundos")
print()

# Calcular cuál fue más rápido
if tiempo_f_cols < tiempo_c_cols:
    speedup = tiempo_c_cols / tiempo_f_cols
    print(f" ✓ F-order fue {speedup:.2f}x MÁS RÁPIDO")
    print(f" → Porque los elementos de cada columna están JUNTOS en memoria")
else:
    speedup = tiempo_f_cols / tiempo_c_cols
    print(f" C-order fue {speedup:.2f}x más rápido (inesperado)")
print()

print("EXPLICACIÓN:")
print(" Array F-order en memoria: [col1][col2][col3]...")
print(" → Sumar col1 accede elementos contiguos: RÁPIDO")
print()
print(" Array C-order en memoria: [fila1][fila2][fila3]...")
print(" → Sumar col1 accede elementos DISPERSOS: LENTO")
print()

# PASO 5: Resumen de resultados
# -------------------------------
print("=" * 70)
print("RESUMEN DE RENDIMIENTO")
print("=" * 70)
print()

print("Operación por FILAS (axis=1):")
print(f" C-order: {tiempo_c_filas:.4f}s {'✓ GANADOR' if tiempo_c_filas < tiempo_f_filas else ''}")
print(f" F-order: {tiempo_f_filas:.4f}s {'✓ GANADOR' if tiempo_f_filas < tiempo_c_filas else ''}")
print()

print("Operación por COLUMNAS (axis=0):")
print(f" C-order: {tiempo_c_cols:.4f}s {'✓ GANADOR' if tiempo_c_cols < tiempo_f_cols else ''}")
print(f" F-order: {tiempo_f_cols:.4f}s {'✓ GANADOR' if tiempo_f_cols < tiempo_c_cols else ''}")
print()

print("=" * 70)
print("CONCLUSIONES Y RECOMENDACIONES")
print("=" * 70)
print()

print("1. El orden de memoria SÍ IMPORTA para el rendimiento")
print(" → Diferencias de 10-50% son comunes en arrays grandes")
print()

print("2. REGLA GENERAL:")
print(" → Si procesas FILAS → usa C-order (default en NumPy)")
print(" → Si procesas COLUMNAS → considera F-order")
print()

print("3. CUÁNDO PREOCUPARSE:")
print(" ✓ Arrays grandes (> 1 millón de elementos)")
print(" ✓ Operaciones repetidas (bucles, entrenamiento ML)")
print(" ✓ Aplicaciones tiempo-real")
print()

print("4. CUÁNDO NO PREOCUPARSE:")
print(" → Arrays pequeños (< 10,000 elementos)")
print(" → Operaciones ocasionales")
print(" → Prototipos y exploración inicial")
print()

print("5. LA MEJOR OPTIMIZACIÓN:")
print(" → Usa NumPy vectorizado (mucho más importante que C vs F)")
print(" → Solo optimiza el orden si ya vectorizaste TODO")
print()

print("=" * 70)

## 3. Optimización y Performance

### 3.1. Vectorización: El Santo Grial de NumPy

#### ¿Qué es la vectorización?

**Vectorización** significa eliminar bucles explícitos (for loops) y usar operaciones de arrays de NumPy que están implementadas en C/Fortran bajo el capó.

#### ¿Por qué Python con bucles es lento?

**Analogía del mundo real - Fábrica de coches:**

**Opción 1: Bucle Python (LENTO)**
- Tienes que hacer 1000 coches
- Cada vez que haces un coche:
  1. Buscas las herramientas (Python busca tipos de datos)
  2. Lees las instrucciones (Python interpreta código)
  3. Montas una pieza
  4. Guardas las herramientas
  5. Repites 1000 veces → MUCHO tiempo perdido en buscar herramientas

**Opción 2: Vectorización NumPy (RÁPIDO)**
- Sacas TODAS las herramientas una sola vez
- Pones todas las piezas en una cinta transportadora
- Una máquina especializada (código en C) procesa las 1000 piezas de golpe
- Todo sucede en memoria contigua, sin buscar nada

**Diferencia de velocidad: 10-100x más rápido**

#### ¿Por qué NumPy vectorizado es tan rápido?

1. **Código compilado:** Las operaciones están escritas en C/Fortran (compilado), no Python (interpretado)
   
2. **Memoria contigua:** Los datos están juntos en memoria, el procesador los puede leer muy rápido (cache hits)

3. **Sin overhead de Python:** No hay que verificar tipos, hacer garbage collection, etc. en cada iteración

4. **Paralelización automática:** NumPy puede usar instrucciones SIMD (Single Instruction Multiple Data) del procesador
   - Ejemplo: Sumar 4 números a la vez en lugar de 1 por 1

5. **Menos código:** Menos líneas = menos bugs = más legible

#### Estrategias de vectorización:

**1. Operaciones element-wise (elemento por elemento):**
```python
# MALO (bucle):
for i in range(len(arr)):
    arr[i] = arr[i] ** 2

# BUENO (vectorizado):
arr = arr ** 2
```

**2. Funciones universales (ufuncs):**
```python
# NumPy tiene funciones optimizadas para todo:
np.sin(), np.cos(), np.exp(), np.log(), np.sqrt(), etc.
```

**3. Broadcasting para evitar bucles anidados:**
```python
# MALO (doble bucle):
for i in range(n):
    for j in range(m):
        result[i, j] = arr1[i] + arr2[j]

# BUENO (broadcasting):
result = arr1[:, np.newaxis] + arr2
```

**4. Funciones de agregación con axis:**
```python
# MALO (bucle por columnas):
for col in range(n_cols):
    suma = 0
    for row in range(n_rows):
        suma += data[row, col]
    resultado[col] = suma

# BUENO (vectorizado):
resultado = np.sum(data, axis=0)
```

**5. Lógica condicional con np.where() o np.select():**
```python
# MALO (bucle con if):
for i in range(len(arr)):
    if arr[i] > 0:
        result[i] = arr[i] * 2
    else:
        result[i] = arr[i] / 2

# BUENO (vectorizado):
result = np.where(arr > 0, arr * 2, arr / 2)
```

#### Ventajas de la vectorización:

| Aspecto | Bucle Python | NumPy Vectorizado |
|---------|--------------|-------------------|
| **Velocidad** | 1x (baseline) | 10-100x más rápido |
| **Legibilidad** | Muchas líneas de código | 1-2 líneas concisas |
| **Bugs** | Más código = más bugs | Menos código = menos bugs |
| **Mantenimiento** | Difícil de modificar | Fácil de modificar |
| **Escalabilidad** | Se vuelve muy lento con datos grandes | Escala bien |

#### ¿Cuándo NO vectorizar?

- Cuando la lógica es muy compleja y la vectorización la hace ilegible
- Cuando solo procesas pocos datos (< 100 elementos) donde no importa la velocidad
- Cuando necesitas operaciones que no se pueden expresar vectorialmente

**En estos casos:** Considera usar Numba (compilación JIT) o Cython en lugar de vectorizar

In [ ]:
# ============================================================
# EJEMPLO 1: Comparar bucle Python vs vectorización NumPy
# ============================================================
#
# TAREA: Calcular sqrt(x) * 2 para cada elemento de un array
#
# Vamos a comparar DOS enfoques:
# 1. Bucle Python tradicional (LENTO)
# 2. Vectorización NumPy (RÁPIDO)
# ============================================================

print("=" * 70)
print("COMPARACIÓN: BUCLE vs VECTORIZACIÓN")
print("=" * 70)
print()

# Crear un array grande para ver la diferencia de velocidad
n = 1_000_000 # 1 millón de elementos
data = np.arange(1, n + 1, dtype=float)

print(f"Array con {n:,} elementos")
print(f"Primeros 10: {data[:10]}")
print()

# ============================================================
# MÉTODO 1: Bucle Python (LENTO)
# ============================================================
print("MÉTODO 1: Bucle Python")
print("-" * 70)

# Crear array vacío para resultados
resultado_bucle = np.zeros(n)

# Medir tiempo de inicio
inicio = time.time()

# BUCLE: Procesar elemento por elemento
for i in range(n):
    # Para cada elemento:
    # 1. Calcular raíz cuadrada
    # 2. Multiplicar por 2
    # 3. Guardar en resultado
    resultado_bucle[i] = np.sqrt(data[i]) * 2

# Medir tiempo final
tiempo_bucle = time.time() - inicio

print(f"Tiempo: {tiempo_bucle:.4f} segundos")
print(f"Primeros 10 resultados: {resultado_bucle[:10]}")
print()

print("¿Qué hace Python en cada iteración del bucle?")
print(" 1. Verificar el tipo de data[i]")
print(" 2. Llamar a la función sqrt() (búsqueda en diccionario)")
print(" 3. Verificar el tipo del resultado")
print(" 4. Hacer la multiplicación")
print(" 5. Verificar tipos de nuevo")
print(" 6. Asignar al array resultado")
print(" → TODO esto 1 millón de veces = LENTO")
print()

# ============================================================
# MÉTODO 2: Vectorización NumPy (RÁPIDO)
# ============================================================
print("MÉTODO 2: Vectorización NumPy")
print("-" * 70)

# Medir tiempo
inicio = time.time()

# UNA SOLA LÍNEA: NumPy procesa todo el array de golpe
resultado_vectorizado = np.sqrt(data) * 2

tiempo_vectorizado = time.time() - inicio

print(f"Tiempo: {tiempo_vectorizado:.4f} segundos")
print(f"Primeros 10 resultados: {resultado_vectorizado[:10]}")
print()

print("¿Qué hace NumPy internamente?")
print(" 1. Identifica que todos los elementos son float64 (una sola vez)")
print(" 2. Llama a una función de C altamente optimizada")
print(" 3. Procesa múltiples elementos en paralelo (SIMD)")
print(" 4. Escribe resultados en memoria contigua")
print(" → TODO el array de 1 vez = MUY RÁPIDO")
print()

# ============================================================
# COMPARACIÓN Y CONCLUSIONES
# ============================================================
print("=" * 70)
print("COMPARACIÓN DE RESULTADOS")
print("=" * 70)

# Verificar que ambos métodos dan el mismo resultado
son_iguales = np.allclose(resultado_bucle, resultado_vectorizado)
print(f"¿Los resultados son iguales? {son_iguales} ✓")
print()

# Calcular la aceleración
aceleracion = tiempo_bucle / tiempo_vectorizado

print("DIFERENCIA DE VELOCIDAD:")
print("-" * 70)
print(f"Tiempo bucle: {tiempo_bucle:.4f} segundos")
print(f"Tiempo vectorizado: {tiempo_vectorizado:.4f} segundos")
print()
print(f"NumPy es {aceleracion:.1f}x MÁS RÁPIDO que el bucle Python")
print()

# Mostrar lo que significa en tiempo real
if tiempo_bucle > 1:
    print("ANALOGÍA:")
    print(f" Si el bucle tarda {tiempo_bucle:.2f} segundos")
    print(f" NumPy tarda solo {tiempo_vectorizado:.4f} segundos")
    print()
    minutos_bucle = tiempo_bucle / 60
    segundos_vectorizado = tiempo_vectorizado
    print(f" En un proceso que toma 1 hora con bucles,")
    print(f" con NumPy vectorizado tomaría solo {60/aceleracion:.1f} minutos")
print()

print("=" * 70)
print("REGLA DE ORO")
print("=" * 70)
print("SIEMPRE vectoriza tus operaciones NumPy:")
print(" ✓ Código más rápido (10-100x)")
print(" ✓ Código más corto y legible")
print(" ✓ Menos bugs")
print(" ✓ Más fácil de mantener")
print()
print("Evita bucles for/while cuando trabajes con arrays NumPy!")
print("=" * 70)

In [ ]:
# Ejemplo 2: Vectorizar lógica condicional con np.where()

# Tarea: Si valor > 0.5, multiplicar por 2; sino, dividir por 2

n = 500_000
data = np.random.rand(n)

# Método 1: Bucle con condicional (LENTO)
inicio = time.time()
resultado_bucle = np.zeros(n)
for i in range(n):
    if data[i] > 0.5:
        resultado_bucle[i] = data[i] * 2
    else:
        resultado_bucle[i] = data[i] / 2
tiempo_bucle = time.time() - inicio

# Método 2: np.where() vectorizado (RÁPIDO)
inicio = time.time()
resultado_vectorizado = np.where(data > 0.5, data * 2, data / 2)
tiempo_vectorizado = time.time() - inicio

print(f"Tiempo con bucle: {tiempo_bucle:.4f} segundos")
print(f"Tiempo con np.where(): {tiempo_vectorizado:.4f} segundos")
print(f"Aceleración: {tiempo_bucle / tiempo_vectorizado:.1f}x más rápido")
print()
print(f"Resultados iguales: {np.allclose(resultado_bucle, resultado_vectorizado)}")

In [ ]:
# Ejemplo 3: Vectorizar cálculos de distancias (común en ML)

# Tarea: Calcular distancias euclidianas entre todos los pares de puntos

n_puntos = 500
puntos = np.random.rand(n_puntos, 2) # 500 puntos en 2D

# Método 1: Doble bucle (MUY LENTO)
inicio = time.time()
distancias_bucle = np.zeros((n_puntos, n_puntos))
for i in range(n_puntos):
    for j in range(n_puntos):
        diff = puntos[i] - puntos[j]
        distancias_bucle[i, j] = np.sqrt(np.sum(diff ** 2))
tiempo_bucle = time.time() - inicio

# Método 2: Broadcasting vectorizado (RÁPIDO)
inicio = time.time()
# Expandir dimensiones para broadcasting: (500, 1, 2) - (1, 500, 2) = (500, 500, 2)
diff = puntos[:, np.newaxis, :] - puntos[np.newaxis, :, :]
distancias_vectorizado = np.sqrt(np.sum(diff ** 2, axis=2))
tiempo_vectorizado = time.time() - inicio

print(f"Tiempo con doble bucle: {tiempo_bucle:.4f} segundos")
print(f"Tiempo vectorizado: {tiempo_vectorizado:.4f} segundos")
print(f"Aceleración: {tiempo_bucle / tiempo_vectorizado:.1f}x más rápido")
print()
print(f"Resultados iguales: {np.allclose(distancias_bucle, distancias_vectorizado)}")
print()
print(f"Matriz de distancias shape: {distancias_vectorizado.shape}")
print(f"Distancia de punto 0 a punto 1: {distancias_vectorizado[0, 1]:.4f}")

### 3.2. Funciones Universales (ufuncs) Personalizadas

NumPy permite crear tus propias funciones universales con `np.vectorize()` o `np.frompyfunc()`. Sin embargo:

** Advertencia:** `np.vectorize()` es principalmente para conveniencia, NO para rendimiento. Internamente sigue usando bucles Python.

**Alternativa mejor:** Usar Numba (compilación JIT) o escribir en C/Cython para funciones críticas.

In [ ]:
# Ejemplo: Crear una función vectorizada personalizada

def clasificar_temperatura(temp: float) -> str:
    """Clasifica una temperatura en categorías."""
    if temp < 10:
        return 'Frío'
    elif temp < 25:
        return 'Templado'
    else:
        return 'Calor'

# Vectorizar la función
clasificar_vectorizado = np.vectorize(clasificar_temperatura)

# Aplicar a un array
temperaturas = np.array([5, 15, 30, 8, 22, 28, 12, 35])
categorias = clasificar_vectorizado(temperaturas)

print(f"Temperaturas: {temperaturas}")
print(f"Categorías: {categorias}")
print()
print("NOTA: Para mejor rendimiento, usa np.select() o np.where() en lugar de np.vectorize()")

In [ ]:
# Versión optimizada usando np.select()

def clasificar_temperatura_optimizado(temps: np.ndarray) -> np.ndarray:
    """Versión optimizada usando np.select()."""
    condiciones = [
        temps < 10,
        (temps >= 10) & (temps < 25),
        temps >= 25
    ]
    categorias = ['Frio', 'Templado', 'Calor']
    # El valor por defecto hay que darlo: sin el, np.select usa 0 y no
    # puede mezclar un entero con una lista de textos.
    return np.select(condiciones, categorias, default='Sin dato')

# Comparar rendimiento con array grande
n = 100_000
temps_grandes = np.random.uniform(-10, 40, n)

inicio = time.time()
resultado_vectorize = clasificar_vectorizado(temps_grandes)
tiempo_vectorize = time.time() - inicio

inicio = time.time()
resultado_select = clasificar_temperatura_optimizado(temps_grandes)
tiempo_select = time.time() - inicio

print(f"Tiempo con np.vectorize: {tiempo_vectorize:.4f} segundos")
print(f"Tiempo con np.select: {tiempo_select:.4f} segundos")
print(f"np.select es {tiempo_vectorize / tiempo_select:.1f}x más rápido")

## 4. Broadcasting Avanzado

### 4.1. Broadcasting Multidimensional

#### ¿Qué es broadcasting? (Repaso rápido)

**Broadcasting** es el mecanismo de NumPy para operar con arrays de **diferentes formas** sin necesidad de copiar datos.

**Analogía del mundo real - Megafonía:**

Imagina una sala con 100 personas en 10 filas de 10:
- **Sin broadcasting:** Tienes que ir persona por persona y decirles el mensaje → 100 veces
- **Con broadcasting:** Usas un megáfono y dices el mensaje UNA VEZ → todos lo escuchan

En NumPy:
```python
# Sin broadcasting (LENTO):
for i in range(100):
    array[i] = array[i] + 5

# Con broadcasting (RÁPIDO):
array = array + 5 # El "5" se "transmite" a todos los elementos
```

#### Reglas de Broadcasting (IMPORTANTES)

NumPy compara las dimensiones de los arrays **de derecha a izquierda**:

**Regla 1:** Si los arrays tienen diferente número de dimensiones, se añaden dimensiones de tamaño 1 a la **izquierda** del array más pequeño.

**Regla 2:** Las dimensiones son compatibles si:
- Son **iguales**, O
- Una de ellas es **1**

**Regla 3:** Las dimensiones de tamaño 1 se **"estiran"** para coincidir con la otra dimensión.

#### Ejemplos paso a paso:

**Ejemplo 1: Sumar escalar a array**
```python
a = np.array([1, 2, 3]) # Shape: (3,)
b = 5 # Shape: () (escalar)

# Broadcasting:
# b se convierte en [5, 5, 5] (conceptualmente, sin copiar)
# Resultado: [1+5, 2+5, 3+5] = [6, 7, 8]
```

**Ejemplo 2: Array 1D + Array 2D**
```python
a = np.array([[1, 2, 3], # Shape: (2, 3)
              [4, 5, 6]])
b = np.array([10, 20, 30]) # Shape: (3,)

# Paso 1: Alinear dimensiones (derecha a izquierda)
# a: (2, 3)
# b: ( 3) → se convierte en (1, 3)

# Paso 2: Broadcasting
# a: (2, 3)
# b: (1, 3) → se "estira" a (2, 3)
# b pasa a ser [[10, 20, 30],
# [10, 20, 30]]

# Resultado:
# [[1+10, 2+20, 3+30], = [[11, 22, 33],
# [4+10, 5+20, 6+30]] [14, 25, 36]]
```

**Ejemplo 3: Arrays incompatibles**
```python
a = np.array([[1, 2, 3], # Shape: (2, 3)
              [4, 5, 6]])
b = np.array([10, 20]) # Shape: (2,)

# Intentar alinear:
# a: (2, 3)
# b: ( 2) → se convierte en (1, 2)
# a: (2, 3)
# b: (1, 2) → ¡3 ≠ 2! → ERROR

# Solución: Añadir dimensión manualmente
b = b[:, np.newaxis] # Shape: (2, 1)
# Ahora:
# a: (2, 3)
# b: (2, 1) → se estira a (2, 3) ✓
```

#### Visualización de Broadcasting:

```
CASO 1: Array (3, 4) + Array (4,)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Array A: (3, 4) Array B: (4,) → se expande a (1, 4) → se estira a (3, 4)
┌─────────────┐ ┌───────────┐ ┌─────────────┐
│ 1 2 3 4 │ │ 10 20 30 40│ │ 10 20 30 40 │
│ 5 6 7 8 │ + └───────────┘ → │ 10 20 30 40 │
│ 9 10 11 12 │ │ 10 20 30 40 │
└─────────────┘ └─────────────┘

Resultado: A + B (elemento por elemento)
┌─────────────┐
│ 11 22 33 44 │
│ 15 26 37 48 │
│ 19 30 41 52 │
└─────────────┘


CASO 2: Array (3, 4) + Array (3, 1)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Array A: (3, 4) Array B: (3, 1) → se estira a (3, 4)
┌─────────────┐ ┌───┐ ┌─────────────┐
│ 1 2 3 4 │ │100│ │100 100 100 100│
│ 5 6 7 8 │ + │200│ → │200 200 200 200│
│ 9 10 11 12 │ │300│ │300 300 300 300│
└─────────────┘ └───┘ └─────────────┘

Resultado: A + B
┌─────────────┐
│101 102 103 104│
│205 206 207 208│
│309 310 311 312│
└─────────────┘
```

#### Aplicaciones comunes de broadcasting:

**1. Normalizar datos por columna (features):**
```python
# Dataset: 1000 muestras × 10 features
data = np.random.rand(1000, 10)

# Calcular media por columna: shape (10,)
mean = np.mean(data, axis=0)

# Restar media (broadcasting automático)
data_centered = data - mean # (1000, 10) - (10,) → funciona ✓
```

**2. Normalizar imágenes en un batch:**
```python
# Batch: 32 imágenes de 64×64×3 (RGB)
batch = np.random.rand(32, 64, 64, 3)

# Media por canal RGB: shape (3,) → se expande a (1, 1, 1, 3)
mean_rgb = np.mean(batch, axis=(0, 1, 2)) # Shape: (3,)

# Normalizar (broadcasting)
batch_normalized = batch - mean_rgb # (32,64,64,3) - (3,) → funciona ✓
```

**3. Crear meshgrid para gráficos:**
```python
x = np.array([1, 2, 3]) # Shape: (3,)
y = np.array([10, 20]) # Shape: (2,)

# Broadcasting para crear grid
X = x[np.newaxis, :] # Shape: (1, 3)
Y = y[:, np.newaxis] # Shape: (2, 1)

grid = X + Y # Shape: (2, 3)
# [[11, 12, 13],
# [21, 22, 23]]
```

In [ ]:
# ============================================================
# EJEMPLO PRÁCTICO: Normalización de batch de imágenes
# ============================================================
#
# CONTEXTO: En deep learning, procesamos imágenes en "batches"
# (lotes). Cada batch contiene múltiples imágenes.
#
# PROBLEMA: Queremos normalizar cada canal (R, G, B)
# independientemente para todo el batch
#
# SOLUCIÓN: Broadcasting nos permite hacerlo SIN bucles
# ============================================================

print("=" * 70)
print("BROADCASTING EN DEEP LEARNING: NORMALIZACIÓN DE IMÁGENES")
print("=" * 70)
print()

# PASO 1: Simular un batch de imágenes
# --------------------------------------
print("PASO 1: Crear batch de imágenes RGB")
print()

# Dimensiones típicas en deep learning:
batch_size = 32 # 32 imágenes por batch
altura = 64 # 64 píxeles de alto
ancho = 64 # 64 píxeles de ancho
canales = 3 # 3 canales (R, G, B)

# Simular imágenes con valores 0-255
batch_imagenes = np.random.default_rng(42).integers(0, 256,
    size=(batch_size, altura, ancho, canales)).astype(np.float32)

print(f"Batch de imágenes:")
print(f" Shape: {batch_imagenes.shape}")
print(f" Interpretación:")
print(f" - Dimensión 0 (batch): {batch_size} imágenes")
print(f" - Dimensión 1 (altura): {altura} píxeles")
print(f" - Dimensión 2 (ancho): {ancho} píxeles")
print(f" - Dimensión 3 (canales): {canales} canales (R, G, B)")
print()
print(f" Total de números: {batch_imagenes.size:,}")
print(f" Rango de valores: [{batch_imagenes.min():.1f}, {batch_imagenes.max():.1f}]")
print()

# PASO 2: Calcular estadísticas por canal
# -----------------------------------------
print("=" * 70)
print("PASO 2: Calcular media y std por canal RGB")
print("=" * 70)
print()

print("OBJETIVO: Queremos estadísticas GLOBALES por canal:")
print(" - Media del canal ROJO en todas las imágenes")
print(" - Media del canal VERDE en todas las imágenes")
print(" - Media del canal AZUL en todas las imágenes")
print()

# Calcular media por canal
# Promediamos sobre: batch (axis=0), altura (axis=1), ancho (axis=2)
# Mantenemos: canales (axis=3)
media_por_canal = np.mean(batch_imagenes, axis=(0, 1, 2), keepdims=True)
std_por_canal = np.std(batch_imagenes, axis=(0, 1, 2), keepdims=True)

print(f"Media por canal:")
print(f" Shape: {media_por_canal.shape}")
print(f" Valores: {media_por_canal.squeeze()}")
print()
print(f" Interpretación:")
print(f" - Rojo (canal 0): media = {media_por_canal[0,0,0,0]:.2f}")
print(f" - Verde (canal 1): media = {media_por_canal[0,0,0,1]:.2f}")
print(f" - Azul (canal 2): media = {media_por_canal[0,0,0,2]:.2f}")
print()

print(f"Desviación estándar por canal:")
print(f" Shape: {std_por_canal.shape}")
print(f" Valores: {std_por_canal.squeeze()}")
print()
print(f" Interpretación:")
print(f" - Rojo (canal 0): std = {std_por_canal[0,0,0,0]:.2f}")
print(f" - Verde (canal 1): std = {std_por_canal[0,0,0,1]:.2f}")
print(f" - Azul (canal 2): std = {std_por_canal[0,0,0,2]:.2f}")
print()

# PASO 3: Broadcasting para normalizar
# --------------------------------------
print("=" * 70)
print("PASO 3: Normalizar usando BROADCASTING")
print("=" * 70)
print()

print("OPERACIÓN: (batch - media) / std")
print()
print("Análisis de shapes para broadcasting:")
print(f" batch_imagenes: {batch_imagenes.shape} ← (32, 64, 64, 3)")
print(f" media_por_canal: {media_por_canal.shape} ← ( 1, 1, 1, 3)")
print(f" std_por_canal: {std_por_canal.shape} ← ( 1, 1, 1, 3)")
print()

print("Broadcasting paso a paso:")
print()
print("1. Comparar dimensiones (derecha a izquierda):")
print(" batch: (32, 64, 64, 3)")
print(" media: ( 1, 1, 1, 3)")
print(" ↑ ↑ ↑ ↑")
print(" ✓ ✓ ✓ ✓ Todas compatibles (32 vs 1, 64 vs 1, ...)")
print()

print("2. Broadcasting automático:")
print(" media shape ( 1, 1, 1, 3) se 'estira' a (32, 64, 64, 3)")
print(" → El mismo vector [media_R, media_G, media_B]")
print(" se aplica a TODOS los píxeles de TODAS las imágenes")
print()

# Realizar la normalización
batch_normalizado = (batch_imagenes - media_por_canal) / std_por_canal

print(f"Resultado:")
print(f" batch_normalizado shape: {batch_normalizado.shape}")
print()

# PASO 4: Verificar la normalización
# ------------------------------------
print("=" * 70)
print("PASO 4: Verificar que la normalización funcionó")
print("=" * 70)
print()

# Calcular nuevas estadísticas
nueva_media = np.mean(batch_normalizado, axis=(0, 1, 2))
nueva_std = np.std(batch_normalizado, axis=(0, 1, 2))

print("Estadísticas DESPUÉS de normalizar:")
print()
print("Media por canal (debería ser ~0):")
print(f" Rojo: {nueva_media[0]:.6f}")
print(f" Verde: {nueva_media[1]:.6f}")
print(f" Azul: {nueva_media[2]:.6f}")
print()

print("Std por canal (debería ser ~1):")
print(f" Rojo: {nueva_std[0]:.6f}")
print(f" Verde: {nueva_std[1]:.6f}")
print(f" Azul: {nueva_std[2]:.6f}")
print()

print("✓ Normalización exitosa: media ≈ 0, std ≈ 1 para cada canal")
print()

# PASO 5: Comparación SIN broadcasting (para entender la ventaja)
# -----------------------------------------------------------------
print("=" * 70)
print("PASO 5: ¿Cómo sería SIN broadcasting? (COMPARACIÓN)")
print("=" * 70)
print()

print("OPCIÓN 1: Sin broadcasting (MUCHOS bucles):")
print()
print("batch_normalizado_manual = np.zeros_like(batch_imagenes)")
print("for b in range(batch_size): # Para cada imagen")
print(" for h in range(altura): # Para cada fila")
print(" for w in range(ancho): # Para cada columna")
print(" for c in range(canales): # Para cada canal")
print(" batch_normalizado_manual[b,h,w,c] = \\")
print(" (batch_imagenes[b,h,w,c] - media[c]) / std[c]")
print()
print("¡4 bucles anidados! Para 32×64×64×3 = 393,216 iteraciones")
print()

print("OPCIÓN 2: Con broadcasting (1 línea):")
print()
print("batch_normalizado = (batch_imagenes - media_por_canal) / std_por_canal")
print()
print("¡Una sola operación! NumPy lo hace todo internamente en C")
print()

# Comparar velocidad
print("=" * 70)
print("COMPARACIÓN DE VELOCIDAD")
print("=" * 70)
print()

# Versión con bucles
inicio = time.time()
batch_normalizado_bucles = np.zeros_like(batch_imagenes)
media_flat = media_por_canal.squeeze()
std_flat = std_por_canal.squeeze()
for b in range(batch_size):
    for h in range(altura):
        for w in range(ancho):
            for c in range(canales):
                batch_normalizado_bucles[b,h,w,c] = \
                    (batch_imagenes[b,h,w,c] - media_flat[c]) / std_flat[c]
tiempo_bucles = time.time() - inicio

# Versión con broadcasting
inicio = time.time()
batch_normalizado_broadcast = (batch_imagenes - media_por_canal) / std_por_canal
tiempo_broadcast = time.time() - inicio

print(f"Tiempo con 4 bucles: {tiempo_bucles:.4f} segundos")
print(f"Tiempo con broadcasting: {tiempo_broadcast:.4f} segundos")
print()
print(f"Broadcasting es {tiempo_bucles/tiempo_broadcast:.1f}x MÁS RÁPIDO")
print()

# Verificar que dan el mismo resultado
son_iguales = np.allclose(batch_normalizado_bucles, batch_normalizado_broadcast)
print(f"¿Los resultados son iguales? {son_iguales} ✓")
print()

print("=" * 70)
print("CONCLUSIÓN")
print("=" * 70)
print()
print("Broadcasting nos permite:")
print(" ✓ Código 10-100x más rápido")
print(" ✓ Código mucho más legible (1 línea vs 4 bucles)")
print(" ✓ Menos errores (menos código = menos bugs)")
print(" ✓ Operaciones en memoria contigua (mejor cache)")
print()
print("Aplicación en Deep Learning:")
print(" → Normalización de batches (como este ejemplo)")
print(" → Data augmentation")
print(" → Operaciones de convolución")
print(" → Batch normalization")
print("=" * 70)

In [ ]:
# Ejemplo 2: Broadcasting con diferentes dimensiones

# Tres arrays de diferentes formas
a = np.arange(12).reshape(3, 4) # Shape: (3, 4)
b = np.arange(3).reshape(3, 1) # Shape: (3, 1)
c = np.arange(4) # Shape: (4,)

print("Array a (3x4):")
print(a)
print(f"Shape: {a.shape}")
print()

print("Array b (3x1):")
print(b)
print(f"Shape: {b.shape}")
print()

print("Array c (1D):")
print(c)
print(f"Shape: {c.shape}")
print()

# Operación con broadcasting: a + b + c
# Broadcasting alinea automáticamente:
# a: (3, 4)
# b: (3, 1) → se expande a (3, 4)
# c: (4,) → se interpreta como (1, 4) → se expande a (3, 4)
resultado = a + b + c

print("Resultado a + b + c:")
print(resultado)
print(f"Shape: {resultado.shape}")

### 4.2. newaxis y expand_dims

Para controlar broadcasting, a menudo necesitamos añadir dimensiones. Hay dos formas principales:

1. **`np.newaxis`** (alias de `None`): Añade una nueva dimensión en la posición especificada
2. **`np.expand_dims(array, axis)`**: Más explícito, hace lo mismo

In [ ]:
# Comparar np.newaxis y np.expand_dims

arr = np.array([1, 2, 3]) # Shape: (3,)
print(f"Array original shape: {arr.shape}")
print(arr)
print()

# Añadir dimensión al final (convertir en columna)
columna_v1 = arr[:, np.newaxis] # Shape: (3, 1)
columna_v2 = np.expand_dims(arr, axis=1) # Shape: (3, 1)

print(f"Como columna (v1) shape: {columna_v1.shape}")
print(columna_v1)
print()

print(f"Como columna (v2) shape: {columna_v2.shape}")
print(columna_v2)
print()

# Añadir dimensión al principio (convertir en fila)
fila_v1 = arr[np.newaxis, :] # Shape: (1, 3)
fila_v2 = np.expand_dims(arr, axis=0) # Shape: (1, 3)

print(f"Como fila (v1) shape: {fila_v1.shape}")
print(fila_v1)
print()

print(f"Como fila (v2) shape: {fila_v2.shape}")
print(fila_v2)

## 5. Casos de Uso en IA

### 5.1. Procesamiento de Imágenes

Las imágenes son arrays 3D (altura × ancho × canales). NumPy es la herramienta fundamental para procesarlas antes de pasarlas a redes neuronales.

In [ ]:
# Simular una imagen RGB (100x100 píxeles, 3 canales)
rng = np.random.default_rng(seed=42)
imagen = rng.integers(0, 256, size=(100, 100, 3), dtype=np.uint8)

print(f"Imagen shape: {imagen.shape}")
print(f"Tipo de datos: {imagen.dtype}")
print(f"Rango de valores: [{imagen.min()}, {imagen.max()}]")
print()

# Operación 1: Convertir a escala de grises (promedio de canales)
gris = np.mean(imagen, axis=2).astype(np.uint8)
print(f"Escala de grises shape: {gris.shape}")
print()

# Operación 2: Normalizar a [0, 1] (común en deep learning)
imagen_normalizada = imagen.astype(np.float32) / 255.0
print(f"Imagen normalizada rango: [{imagen_normalizada.min():.2f}, {imagen_normalizada.max():.2f}]")
print()

# Operación 3: Extraer canal rojo
canal_rojo = imagen[:, :, 0]
print(f"Canal rojo shape: {canal_rojo.shape}")
print()

# Operación 4: Flip horizontal (augmentation)
imagen_flip = np.fliplr(imagen)
print(f"Imagen con flip horizontal shape: {imagen_flip.shape}")

In [ ]:
# Visualizar transformaciones
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(imagen)
axes[0].set_title('Original RGB')
axes[0].axis('off')

axes[1].imshow(gris, cmap='gray')
axes[1].set_title('Escala de Grises')
axes[1].axis('off')

axes[2].imshow(canal_rojo, cmap='Reds')
axes[2].set_title('Canal Rojo')
axes[2].axis('off')

axes[3].imshow(imagen_flip)
axes[3].set_title('Flip Horizontal')
axes[3].axis('off')

plt.tight_layout()
plt.show()

### 5.2. Preparación de Datos para Machine Learning

Antes de entrenar modelos, los datos deben transformarse. NumPy proporciona todas las herramientas necesarias.

In [ ]:
# Generar dataset sintético
rng = np.random.default_rng(seed=42)

n_muestras = 1000
n_features = 5

# Features con diferentes escalas
X = np.column_stack([
    rng.normal(100, 15, n_muestras), # Feature 1: ~100
    rng.normal(0.5, 0.1, n_muestras), # Feature 2: ~0.5
    rng.uniform(0, 1000, n_muestras), # Feature 3: 0-1000
    rng.exponential(2, n_muestras), # Feature 4: exponencial
    rng.integers(1, 10, n_muestras) # Feature 5: enteros 1-10
])

print(f"Dataset shape: {X.shape}")
print(f"Primeras 5 muestras:\n{X[:5]}")
print()

# Estadísticas por feature
print("Estadísticas por feature:")
print(f"Media: {np.mean(X, axis=0)}")
print(f"Std: {np.std(X, axis=0)}")
print(f"Min: {np.min(X, axis=0)}")
print(f"Max: {np.max(X, axis=0)}")

In [ ]:
# Normalización StandardScaler (media=0, std=1)
def standard_scaler(X: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Normaliza features: (X - media) / std
    
    Returns:
        X_scaled, media, std
    """
    media = np.mean(X, axis=0)
    std = np.std(X, axis=0)
    X_scaled = (X - media) / std
    return X_scaled, media, std

X_standard, media, std = standard_scaler(X)

print("Después de StandardScaler:")
print(f"Nueva media: {np.mean(X_standard, axis=0)}")
print(f"Nueva std: {np.std(X_standard, axis=0)}")
print()

In [ ]:
# Normalización MinMaxScaler (rango [0, 1])
def minmax_scaler(X: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Normaliza features a rango [0, 1]: (X - min) / (max - min)
    
    Returns:
        X_scaled, min_vals, max_vals
    """
    min_vals = np.min(X, axis=0)
    max_vals = np.max(X, axis=0)
    X_scaled = (X - min_vals) / (max_vals - min_vals)
    return X_scaled, min_vals, max_vals

X_minmax, min_vals, max_vals = minmax_scaler(X)

print("Después de MinMaxScaler:")
print(f"Nueva min: {np.min(X_minmax, axis=0)}")
print(f"Nueva max: {np.max(X_minmax, axis=0)}")
print()

In [ ]:
# Visualizar efecto de normalización
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Dataset original
axes[0].boxplot(X, tick_labels=[f'F{i+1}' for i in range(n_features)])
axes[0].set_title('Datos Originales (diferentes escalas)')
axes[0].set_ylabel('Valor')
axes[0].grid(True, alpha=0.3)

# StandardScaler
axes[1].boxplot(X_standard, tick_labels=[f'F{i+1}' for i in range(n_features)])
axes[1].set_title('StandardScaler (media=0, std=1)')
axes[1].set_ylabel('Valor')
axes[1].grid(True, alpha=0.3)

# MinMaxScaler
axes[2].boxplot(X_minmax, tick_labels=[f'F{i+1}' for i in range(n_features)])
axes[2].set_title('MinMaxScaler (rango [0, 1])')
axes[2].set_ylabel('Valor')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Observa cómo la normalización coloca todas las features en escalas comparables.")

### 5.3. Batch Processing para Deep Learning

En deep learning, procesamos datos en batches (lotes). NumPy facilita la creación y manipulación de batches.

In [ ]:
def crear_batches(X: np.ndarray, y: np.ndarray, batch_size: int, shuffle: bool = True) -> list:
    """
    Divide dataset en batches.
    
    Args:
        X: Features (n_samples, n_features)
        y: Labels (n_samples,)
        batch_size: Tamaño de cada batch
        shuffle: Si True, mezcla los datos antes de crear batches
        
    Returns:
        Lista de tuplas (X_batch, y_batch)
    """
    n_samples = X.shape[0]
    indices = np.arange(n_samples)
    
    if shuffle:
        rng = np.random.default_rng()
        rng.shuffle(indices)
    
    batches = []
    for start_idx in range(0, n_samples, batch_size):
        end_idx = min(start_idx + batch_size, n_samples)
        batch_indices = indices[start_idx:end_idx]
        batches.append((X[batch_indices], y[batch_indices]))
    
    return batches

# Generar datos de ejemplo
n_samples = 1000
n_features = 10
X = np.random.randn(n_samples, n_features)
y = np.random.randint(0, 2, n_samples) # Clasificación binaria

# Crear batches
batch_size = 32
batches = crear_batches(X, y, batch_size, shuffle=True)

print(f"Dataset: {X.shape[0]} muestras")
print(f"Batch size: {batch_size}")
print(f"Número de batches: {len(batches)}")
print()

# Inspeccionar primer batch
X_batch, y_batch = batches[0]
print(f"Primer batch:")
print(f" X_batch shape: {X_batch.shape}")
print(f" y_batch shape: {y_batch.shape}")
print(f" y_batch (primeros 10): {y_batch[:10]}")

## Ejercicios Prácticos

### Instrucciones

Completa los siguientes ejercicios para consolidar tu comprensión de NumPy avanzado:

- **Básicos (1-4):** Aplicación de conceptos avanzados
- **Intermedios (5-8):** Combinación de técnicas
- **Avanzados (9-12):** Problemas complejos de IA/ML

### Ejercicio 1 (Básico): Compresión de Imagen con SVD

**¿Para qué sirve?**
SVD (Descomposición en Valores Singulares) es una técnica de álgebra lineal que nos permite comprimir imágenes manteniendo la información más importante. En compresión de imágenes, los valores singulares grandes contienen la mayor parte de la información visual, mientras que los pequeños representan detalles menos importantes que podemos descartar.

**Aplicación real:** Netflix y YouTube usan variantes de SVD para comprimir videos y reducir el ancho de banda.

**Tu tarea:**
Vas a simular cómo funciona la compresión de imágenes usando solo los valores singulares más importantes.

**Pasos a seguir:**

1. **Genera una "imagen" sintética** (matriz 50×50 con valores aleatorios)
2. **Aplica SVD** para descomponerla en U, Σ, V^T
3. **Reconstruye la imagen usando solo k=10 valores singulares** (en vez de los 50 originales)
4. **Calcula el error** de reconstrucción (diferencia con la original)
5. **Visualiza** la imagen original vs la comprimida

**Código base para empezar:**
```python
import numpy as np
import matplotlib.pyplot as plt

# Paso 1: Generar imagen sintética 50×50
rng = np.random.default_rng(42)
imagen_original = rng.random((50, 50))

# Paso 2: Aplicar SVD
# PISTA: np.linalg.svd(imagen, full_matrices=False)
U, s, Vt = ... # TODO: Completa aquí

print(f"Forma de U: {U.shape}") # Debe ser (50, 50)
print(f"Forma de s: {s.shape}") # Debe ser (50,)
print(f"Forma de Vt: {Vt.shape}") # Debe ser (50, 50)

# Paso 3: Reconstruir con solo k=10 componentes
k = 10

# PISTA: Para reconstruir, necesitas:
# - Primeras k columnas de U: U[:, :k]
# - Primeros k valores singulares: s[:k]
# - Primeras k filas de Vt: Vt[:k, :]
#
# Reconstrucción = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]

imagen_reconstruida = ... # TODO: Completa la reconstrucción

# Paso 4: Calcular error
error = np.mean(np.abs(imagen_original - imagen_reconstruida))
print(f"\nError promedio: {error:.6f}")
print(f"Error relativo: {error / np.mean(imagen_original) * 100:.2f}%")

# Paso 5: Visualizar comparación
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(imagen_original, cmap='gray')
axes[0].set_title('Imagen Original (50 componentes)')
axes[0].axis('off')

axes[1].imshow(imagen_reconstruida, cmap='gray')
axes[1].set_title(f'Imagen Comprimida ({k} componentes)')
axes[1].axis('off')

plt.tight_layout()
plt.show()

print(f"\n✓ Compresión: {k}/50 componentes = {k/50*100:.0f}% del tamaño original")
print(f"✓ Calidad visual: {(1-error)*100:.1f}% preservada")
```

**Preguntas de reflexión:**
- ¿Qué pasa si usas k=5 en lugar de k=10? ¿Y k=20?
- ¿Por qué crees que funciona mejor mantener los valores singulares grandes en lugar de los pequeños?

**Referencia:** Vuelve a la sección de SVD en el notebook (celdas 6-8) si necesitas recordar cómo funciona la descomposición.

In [ ]:
# TODO: Escribe tu código aquí
# Pista 1: rng.random((50, 50)) para generar imagen
# Pista 2: U, S, Vt = np.linalg.svd(imagen, full_matrices=False)
# Pista 3: Reconstruir con k=10 componentes
# Pista 4: Error = np.linalg.norm(original - reconstruida)



### Ejercicio 2 (Básico): Views vs Copies - Entendiendo la Memoria

**¿Por qué es importante?**
Imagina que estás trabajando con un array de 10GB (ej: un video). Si haces una **copia**, duplicas esos 10GB en memoria → lento y puede quedarse sin RAM. Si usas una **view**, solo creas un "acceso directo" → rápido y eficiente.

**Concepto clave:**

**VIEW (Vista):** Es como un acceso directo a los datos originales. Si modificas la view, modificas el original (comparten la misma memoria).

**COPY (Copia):** Es una copia independiente de los datos. Modificar la copia NO afecta al original (memorias separadas).

```
VISTA: COPIA:
┌──────┐ ┌──────┐ ┌──────┐
│ a │ ← datos originales │ a │ │ c │
└──────┘ └──────┘ └──────┘
   ↑ ↑ ↑
   │ (comparten memoria) │ │
┌──────┐ │ (memorias
│ b │ ← view de a │ separadas)
└──────┘ │
Si cambias b, cambias a Si cambias c,
                                a no cambia
```

**Tu tarea:**
Determina si cada operación crea una VIEW o una COPY, y verifica experimentalmente.

**Plantilla para cada caso:**

```python
import numpy as np

# Crear array original
a = np.arange(20)
print(f"Array original: {a}\n")

# ==========================================
# CASO 1: Slicing básico b = a[5:15]
# ==========================================
b = a[5:15]

# Verificar si es view o copy
print(f"CASO 1: b = a[5:15]")
print(f"¿b es una view de a? {b.base is a}") # True = VIEW, False = COPY
print(f"ID memoria de a: {id(a)}")
print(f"ID memoria de b: {id(b)}")
print(f"¿Comparten memoria? {b.base is a}\n")

# Prueba: modificar b y ver si afecta a a
b[0] = 999
print(f"Después de b[0] = 999:")
print(f"a[5] = {a[5]}") # Si cambió → VIEW, si no cambió → COPY
print(f"Conclusión: {'VIEW' if a[5] == 999 else 'COPY'}\n")

# Resetear para siguiente caso
a = np.arange(20)

# ==========================================
# CASO 2: Fancy indexing c = a[[1, 3, 5, 7]]
# ==========================================
c = a[[1, 3, 5, 7]]

print(f"CASO 2: c = a[[1, 3, 5, 7]]")
print(f"¿c es una view de a? {c.base is a}")
c[0] = 999
print(f"Después de c[0] = 999:")
print(f"a[1] = {a[1]}")
print(f"Conclusión: {'VIEW' if a[1] == 999 else 'COPY'}\n")

# Resetear
a = np.arange(20)

# ==========================================
# CASO 3: Reshape d = a.reshape(4, 5)
# ==========================================
d = a.reshape(4, 5)

# TODO: Completa el análisis para este caso siguiendo la plantilla anterior
print(f"CASO 3: d = a.reshape(4, 5)")
# ... (completa tú)

# ==========================================
# CASO 4: Boolean indexing e = a[a > 10]
# ==========================================
e = a[a > 10]

# TODO: Completa el análisis para este caso
print(f"CASO 4: e = a[a > 10]")
# ... (completa tú)
```

**Resumen de reglas (completa después del experimento):**

| Operación | View o Copy | ¿Por qué? |
|-----------|-------------|-----------|
| Slicing básico `a[5:15]` | ? | ? |
| Fancy indexing `a[[1,3,5]]` | ? | ? |
| Reshape `a.reshape(...)` | ? | ? |
| Boolean indexing `a[a>10]` | ? | ? |

**Reglas generales de NumPy:**
- **Slicing básico** (`:`, rangos) → VIEW
- **Fancy indexing** (listas, arrays) → COPY
- **Boolean indexing** (máscaras) → COPY
- **reshape()** → VIEW (si es posible sin reorganizar memoria)

**Consejo práctico:**
Si necesitas una copia explícita: usa `.copy()` → `b = a[5:15].copy()`
Si necesitas asegurarte que es view: verifica con `b.base is a`

In [ ]:
a = np.arange(20)

# TODO: Realiza cada operación y verifica view vs copy
# Pista: Modifica el resultado y comprueba si 'a' cambió
# Pista: Usa .base para verificar si es view



### Ejercicio 3 (Básico): Vectorizar Función con np.where()

Implementa la función ReLU (Rectified Linear Unit) sin bucles:
- ReLU(x) = max(0, x)
- Si x < 0 → 0, sino → x

Aplícala a un array de 1 millón de números aleatorios entre -10 y 10.

In [ ]:
# TODO: Implementa ReLU vectorizada
# Pista: np.where(condicion, valor_si_true, valor_si_false)
# Pista: Alternativamente, usa np.maximum(0, x)



### Ejercicio 4 (Básico): Broadcasting para Normalización

Dada una matriz de datos 100×5, normaliza cada columna independientemente usando StandardScaler (media=0, std=1) aplicando broadcasting.

In [ ]:
rng = np.random.default_rng(seed=42)
datos = rng.normal(50, 10, size=(100, 5))

# TODO: Normaliza cada columna con broadcasting
# Pista 1: Calcula media y std por columna (axis=0)
# Pista 2: Broadcasting: (100, 5) - (5,) funciona automáticamente



### Ejercicio 5 (Intermedio): PCA desde Cero - Reducción de Dimensionalidad

**¿Qué es PCA y para qué sirve?**

**PCA (Principal Component Analysis)** es una técnica de **reducción de dimensionalidad**. Imagina que tienes datos con 1000 características (columnas), pero muchas están correlacionadas. PCA encuentra las "direcciones" donde hay más variación y te permite representar los datos con menos dimensiones SIN perder mucha información.

**Aplicaciones reales:**
- **Reconocimiento facial:** Reducir imágenes de 10,000 píxeles a 50 componentes principales → 200x más rápido
- **Genómica:** Analizar datos con 20,000 genes → reducir a 10 componentes para visualizar
- **Compresión de datos:** Reducir tamaño de datasets manteniendo 95% de la información

**Analogía:**
Imagina que tienes fotos de personas tomadas desde todos los ángulos. PCA encuentra que "altura" y "peso" capturan el 90% de la variación entre personas, así que puedes describir a alguien con solo esos 2 números en vez de 1000 píxeles.

**Tu tarea:**
Implementar PCA paso a paso para entender cómo funciona internamente.

**Código completo con guía:**

```python
import numpy as np
import matplotlib.pyplot as plt

# ========== PASO 1: Generar datos 2D con correlación ==========
print("PASO 1: Generar datos 2D correlacionados")
print("-" * 50)

np.random.seed(42)

# Definir media y covarianza
mean = [0, 0] # Centro en el origen
cov = [[2.0, 1.5], # Varianza en X = 2, covarianza = 1.5 (correlación positiva)
       [1.5, 1.0]] # Varianza en Y = 1

# Generar 200 puntos con esta distribución
datos = np.random.multivariate_normal(mean, cov, 200)

print(f"Forma de datos: {datos.shape}") # (200, 2)
print(f"Primeras 5 filas:\n{datos[:5]}\n")

# Visualizar datos originales
plt.figure(figsize=(8, 6))
plt.scatter(datos[:, 0], datos[:, 1], alpha=0.6, s=50)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Datos Originales (nota la correlación diagonal)')
plt.grid(True, alpha=0.3)
plt.axis('equal')
plt.show()

# ========== PASO 2: Centrar los datos (restar media) ==========
print("PASO 2: Centrar los datos")
print("-" * 50)

# ¿Por qué centrar? PCA busca direcciones de máxima varianza desde el centro
datos_centrados = datos - np.mean(datos, axis=0)

print(f"Media ANTES de centrar: {np.mean(datos, axis=0)}")
print(f"Media DESPUÉS de centrar: {np.mean(datos_centrados, axis=0)}") # Debe ser ~[0, 0]
print()

# ========== PASO 3: Calcular matriz de covarianza ==========
print("PASO 3: Matriz de covarianza")
print("-" * 50)

# Covarianza mide cómo varían juntas dos variables
# cov[i,j] = correlación entre feature i y feature j
cov_matrix = np.cov(datos_centrados.T) # Transponer porque cov() espera (features, samples)

print(f"Forma de covarianza: {cov_matrix.shape}") # (2, 2)
print(f"Matriz de covarianza:\n{cov_matrix}")
print(f"¿Es simétrica? {np.allclose(cov_matrix, cov_matrix.T)}\n")

# ========== PASO 4: Eigenvalues y Eigenvectors ==========
print("PASO 4: Descomposición en eigenvalues y eigenvectors")
print("-" * 50)

eigenvalues, eigenvectors = np.linalg.eig(cov_matrix)

# Ordenar por eigenvalues (de mayor a menor)
idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]

print(f"Eigenvalues (varianza capturada por cada componente):")
print(eigenvalues)

print(f"\nVarianza explicada por cada componente:")
varianza_explicada = eigenvalues / eigenvalues.sum() * 100
for i, var in enumerate(varianza_explicada):
    print(f" Componente {i+1}: {var:.2f}%")

print(f"\nEigenvectors (direcciones de los componentes):")
print(eigenvectors)
print()

# ========== PASO 5: Proyectar datos en componentes principales ==========
print("PASO 5: Proyección en componentes principales")
print("-" * 50)

# Elegir k componentes (en este caso 2D → 1D, así que k=1)
k = 1

# Proyección: datos_nuevos = datos_centrados @ eigenvectors[:, :k]
datos_proyectados = datos_centrados @ eigenvectors[:, :k]

print(f"Forma de datos proyectados: {datos_proyectados.shape}") # (200, 1) → reducción!
print(f"Primeros 5 datos proyectados:\n{datos_proyectados[:5].flatten()}\n")

# ========== PASO 6: Visualización comparativa ==========
print("PASO 6: Visualización")
print("-" * 50)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Subplot 1: Datos originales con componentes principales
axes[0].scatter(datos_centrados[:, 0], datos_centrados[:, 1], alpha=0.6, s=50, label='Datos')

# Dibujar eigenvectors (componentes principales)
origin = [0, 0]
for i in range(len(eigenvalues)):
    axes[0].arrow(*origin, *(eigenvectors[:, i] * np.sqrt(eigenvalues[i]) * 2),
                  head_width=0.2, head_length=0.3, fc=f'C{i+1}', ec=f'C{i+1}',
                  linewidth=3, label=f'PC{i+1} ({varianza_explicada[i]:.1f}%)')

axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')
axes[0].set_title('Datos Centrados + Componentes Principales')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].axis('equal')

# Subplot 2: Datos proyectados en 1D
axes[1].scatter(datos_proyectados, np.zeros_like(datos_proyectados), alpha=0.6, s=50)
axes[1].set_xlabel('Componente Principal 1')
axes[1].set_ylabel('')
axes[1].set_title(f'Datos Proyectados en 1D ({varianza_explicada[0]:.1f}% varianza)')
axes[1].grid(True, alpha=0.3, axis='x')
axes[1].set_ylim(-0.5, 0.5)

plt.tight_layout()
plt.show()

print(f"\n✓ Reducción dimensional: 2D → 1D")
print(f"✓ Información preservada: {varianza_explicada[0]:.1f}%")
print(f"✓ Información perdida: {100 - varianza_explicada[0]:.1f}%")
```

**Preguntas de reflexión:**
1. ¿Qué observas en la dirección del primer componente principal? ¿Coincide con la diagonal de correlación?
2. Si redujeras de 2D a 1D, ¿qué % de información perderías?
3. ¿En qué aplicaciones sería aceptable perder 10-20% de información a cambio de reducir 1000 dimensiones a 50?

**Conexión con ML:**
Antes de entrenar modelos de ML con muchas features, a menudo aplicamos PCA para:
- Acelerar entrenamiento (menos dimensiones)
- Reducir overfitting (menos parámetros)
- Visualizar datos de alta dimensión

In [ ]:
# TODO: Implementa PCA completo
# Pista 1: Genera datos correlacionados
# Pista 2: Covarianza: np.cov(datos_centrados.T)
# Pista 3: Eigenvalues: np.linalg.eigh(cov_matrix)
# Pista 4: Proyectar: datos_pca = datos_centrados @ eigenvectors[:, :k]



### Ejercicio 6 (Intermedio): Matriz de Distancias Eficiente

Implementa una función vectorizada que calcule la matriz de distancias euclidianas entre todos los pares de puntos en un conjunto de datos.

**Fórmula:** `dist(i, j) = sqrt(sum((x_i - x_j)^2))`

**Restricción:** No usar bucles, solo broadcasting y operaciones vectorizadas.

In [ ]:
def matriz_distancias(puntos: np.ndarray) -> np.ndarray:
    """
    Calcula matriz de distancias euclidianas entre todos los pares de puntos.
    
    Args:
        puntos: Array de shape (n_puntos, n_dimensiones)
        
    Returns:
        Matriz de distancias (n_puntos, n_puntos)
    """
    # TODO: Implementa usando broadcasting
    # Pista 1: Expande puntos a (n, 1, dim) y (1, n, dim)
    # Pista 2: Diferencias: (n, 1, dim) - (1, n, dim) = (n, n, dim)
    # Pista 3: Suma cuadrados en axis=2, luego sqrt
    pass

# Prueba con 100 puntos en 3D
puntos = np.random.randn(100, 3)
# TODO: Calcula y verifica la matriz de distancias



### Ejercicio 7 (Intermedio): Augmentation de Imágenes

Crea funciones vectorizadas para data augmentation de imágenes:
1. Flip horizontal
2. Flip vertical
3. Rotación 90°, 180°, 270°
4. Ajuste de brillo (añadir/restar valor)
5. Ajuste de contraste (multiplicar por factor)

Aplica cada transformación a una imagen sintética y visualiza los resultados.

In [ ]:
# TODO: Implementa funciones de augmentation
# Pista 1: Flip horizontal: np.fliplr(imagen)
# Pista 2: Rotación: np.rot90(imagen, k=1)
# Pista 3: Brillo: np.clip(imagen + delta, 0, 255)
# Pista 4: Contraste: np.clip(imagen * factor, 0, 255)



### Ejercicio 8 (Intermedio): Normalización por Batch

Implementa Batch Normalization:
- Dado un batch de datos (32, 10) [32 muestras, 10 features]
- Normaliza cada feature independientemente: `(x - mean) / sqrt(var + epsilon)`
- Usa epsilon=1e-5 para estabilidad numérica
- Verifica que cada feature tenga media ~0 y varianza ~1

In [ ]:
def batch_normalization(batch: np.ndarray, epsilon: float = 1e-5) -> np.ndarray:
    """
    Aplica Batch Normalization.
    
    Args:
        batch: Array de shape (batch_size, n_features)
        epsilon: Pequeño valor para estabilidad numérica
        
    Returns:
        Batch normalizado
    """
    # TODO: Implementa batch normalization
    # Pista 1: Calcula media y varianza por feature (axis=0)
    # Pista 2: Normaliza: (batch - media) / sqrt(var + epsilon)
    pass

# Prueba
batch = np.random.randn(32, 10) * 5 + 10 # Media ~10, std ~5
# TODO: Aplica batch_norm y verifica resultados



### Ejercicio 9 (Avanzado): Convolución 2D desde Cero

Implementa una convolución 2D (operación fundamental en CNNs) sin usar bucles:
- Entrada: imagen (H, W), kernel (K, K)
- Salida: imagen convolucionada
- Usa `np.lib.stride_tricks.as_strided()` para crear vistas deslizantes

In [ ]:
def convolve2d(imagen: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    """
    Aplica convolución 2D.
    
    Args:
        imagen: Array 2D (H, W)
        kernel: Array 2D (K, K)
        
    Returns:
        Imagen convolucionada
    """
    # TODO: Implementa convolución
    # Este es AVANZADO - requiere entender stride_tricks
    # Pista: Busca "numpy sliding window view" o "as_strided"
    pass

# Prueba con kernel de detección de bordes
imagen_test = np.random.rand(10, 10)
kernel_bordes = np.array([[-1, -1, -1],
                          [-1, 8, -1],
                          [-1, -1, -1]])

# TODO: Aplica convolución y visualiza



### Ejercicio 10 (Avanzado): K-Means Clustering

Implementa el algoritmo K-Means usando solo NumPy:
1. Inicializar K centroides aleatoriamente
2. Asignar cada punto al centroide más cercano
3. Actualizar centroides (media de puntos asignados)
4. Repetir hasta convergencia

**Restricción:** Vectoriza el cálculo de distancias (sin bucles).

In [ ]:
def kmeans(X: np.ndarray, K: int, max_iters: int = 100) -> tuple[np.ndarray, np.ndarray]:
    """
    Algoritmo K-Means.
    
    Args:
        X: Datos (n_samples, n_features)
        K: Número de clusters
        max_iters: Iteraciones máximas
        
    Returns:
        centroides, asignaciones
    """
    # TODO: Implementa K-Means
    # Pista 1: Inicializa centroides con K muestras aleatorias
    # Pista 2: Calcula distancias con broadcasting
    # Pista 3: Asigna con np.argmin(distancias, axis=1)
    # Pista 4: Actualiza centroides: np.mean(X[asignaciones == k], axis=0)
    pass

# Generar datos con 3 clusters
rng = np.random.default_rng(seed=42)
X = np.vstack([
    rng.normal([0, 0], 0.5, (100, 2)),
    rng.normal([3, 3], 0.5, (100, 2)),
    rng.normal([0, 3], 0.5, (100, 2))
])

# TODO: Aplica K-Means con K=3 y visualiza resultados



### Ejercicio 11 (Avanzado): Softmax y Cross-Entropy

Implementa las funciones Softmax y Cross-Entropy Loss (usadas en clasificación):

**Softmax:** `softmax(x_i) = exp(x_i) / sum(exp(x))`
**Cross-Entropy:** `loss = -sum(y_true * log(y_pred))`

**Importante:** Implementa versión numéricamente estable de Softmax.

In [ ]:
def softmax(logits: np.ndarray) -> np.ndarray:
    """
    Calcula Softmax de forma numéricamente estable.
    
    Args:
        logits: Array de shape (n_samples, n_classes)
        
    Returns:
        Probabilidades normalizadas
    """
    # TODO: Implementa softmax estable
    # Pista: Resta el máximo de cada fila antes de exp() para estabilidad
    # logits_shifted = logits - np.max(logits, axis=1, keepdims=True)
    pass

def cross_entropy(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """
    Calcula Cross-Entropy Loss.
    
    Args:
        y_true: One-hot encoded labels (n_samples, n_classes)
        y_pred: Probabilidades predichas (n_samples, n_classes)
        
    Returns:
        Pérdida promedio
    """
    # TODO: Implementa cross-entropy
    # Pista: Añade pequeño epsilon para evitar log(0)
    # loss = -np.sum(y_true * np.log(y_pred + 1e-15)) / len(y_true)
    pass

# Prueba
logits = np.random.randn(10, 5) # 10 muestras, 5 clases
y_true = np.eye(5)[np.random.randint(0, 5, 10)] # One-hot encoded

# TODO: Calcula softmax y cross-entropy



### Ejercicio 12 (Avanzado): Mini-Batch Gradient Descent

Implementa entrenamiento de regresión lineal con mini-batch gradient descent:
1. Genera datos sintéticos y = 3x + 2 + ruido
2. Implementa forward pass (predicción)
3. Calcula MSE loss
4. Calcula gradientes
5. Actualiza pesos con SGD
6. Entrena por epochs con batches
7. Visualiza convergencia del loss

In [ ]:
# TODO: Implementa entrenamiento completo de regresión lineal
# Este ejercicio integra múltiples conceptos:
# - Generación de datos
# - Batching
# - Forward/backward pass
# - Optimización
# - Visualización

# Pista: Gradientes para MSE loss en regresión lineal:
# dL/dw = (2/n) * X^T @ (y_pred - y_true)
# dL/db = (2/n) * sum(y_pred - y_true)



## Resumen de Conceptos Clave

### Álgebra Lineal Avanzada

1. **SVD (Singular Value Decomposition):**
   - Descompone cualquier matriz en U Σ V^T
   - Aplicaciones: PCA, compresión, sistemas de recomendación
   - `U, S, Vt = np.linalg.svd(A)`

2. **Descomposición QR:**
   - A = QR (Q ortogonal, R triangular superior)
   - Más estable que inversión directa
   - Útil para resolver sistemas sobre-determinados

3. **Eigenvalues y Eigenvectors:**
   - Av = λv (v es eigenvector, λ es eigenvalue)
   - Base de PCA, PageRank, análisis espectral
   - `eigenvals, eigenvecs = np.linalg.eig(A)`

4. **Normas:**
   - L1 (Manhattan), L2 (Euclidiana), L∞ (Infinito)
   - Regularización en ML, normalización de embeddings

### Gestión de Memoria

1. **Views vs Copies:**
   - **View:** Misma memoria, modificar afecta al original
   - **Copy:** Nueva memoria, independiente del original
   - Slicing → view, fancy indexing → copy
   - Usa `.copy()` cuando necesites independencia

2. **Memory Layout:**
   - C-order (row-major): más rápido para operaciones por filas
   - F-order (column-major): más rápido para operaciones por columnas

### Optimización y Performance

1. **Vectorización:**
   - Eliminar bucles Python → 10-100x más rápido
   - Usar ufuncs, broadcasting, funciones de agregación
   - `np.where()` para lógica condicional

2. **Broadcasting Avanzado:**
   - Operaciones entre arrays de diferentes formas
   - `np.newaxis` o `np.expand_dims()` para añadir dimensiones
   - Esencial para batch processing

### Aplicaciones en IA

1. **Procesamiento de Imágenes:**
   - Conversión a escala de grises
   - Normalización [0, 1]
   - Transformaciones (flip, rotación)

2. **Preparación de Datos:**
   - StandardScaler: (X - μ) / σ
   - MinMaxScaler: (X - min) / (max - min)
   - Batch processing para deep learning

### Mejores Prácticas

- ✓ Siempre vectoriza antes de optimizar
- ✓ Usa views cuando sea posible para ahorrar memoria
- ✓ Aprovecha broadcasting para evitar bucles
- ✓ Normaliza datos antes de entrenar modelos
- ✓ Usa SVD/QR en lugar de inversión directa para estabilidad
- ✗ Evita `np.vectorize()` para código crítico (usa Numba o C)
- ✗ No copies arrays innecesariamente (usa views)
- ✗ No uses bucles cuando puedas vectorizar

## Recursos Adicionales

### Documentación Oficial
- [NumPy Linear Algebra](https://numpy.org/doc/stable/reference/routines.linalg.html)
- [NumPy Broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html)
- [NumPy Performance Tips](https://numpy.org/doc/stable/user/performance.html)

### Artículos y Tutoriales
- [From Python to Numpy](https://www.labri.fr/perso/nrougier/from-python-to-numpy/) - Nicolas Rougier
- [NumPy Illustrated](https://betterprogramming.pub/numpy-illustrated-the-visual-guide-to-numpy-3b1d4976de1d)
- [Advanced NumPy](https://scipy-lectures.org/advanced/advanced_numpy/)

### Libros
- "High Performance Python" - Micha Gorelick
- "Python for Data Analysis" - Wes McKinney (capítulos de NumPy)

### Herramientas Complementarias
- **Numba:** Compilación JIT para acelerar código NumPy
- **Dask:** NumPy paralelo para datasets que no caben en memoria
- **CuPy:** NumPy en GPU (NVIDIA CUDA)

---

**Notebook creado para el Curso de Especialización en IA y Big Data**
**Módulo:** Programación de Inteligencia Artificial
**Unidad:** UD3 - NumPy y Pandas
**Fecha:** 2025

---